This code is designed to extract grooming bouts from DeepLabCut pose estimation data

1. Multi-Animal object classification
    Generate bounding boxes from high DLC keypoint likelihoods to train a YOLO model to detect monkeys in the grooming videos
2. Tracking with YOLO & Bytetrack
    Run all training video through YOLO to generate bboxes
    Bytetrack to connect bboxes into longer tracks for better monkey identification
    Stitch tracks by spatial positioning to build identity chains
    Assign chains identity A and B
3. Preprocess DLC data and assign A and B to keypoint
    Convert h5 files into csv
    Assign keypoints to A and B bbox IDs
    -> monkey A and B keypoints for consistent individual identification
4. Parse manually annotated grooming bouts
    Parse bout information from training videos:
        start sec, end sec, duration, direction (A grooming B or B grooming A)
5. Feature extraction
    Calculate distances, velocities, angles, likelihoods, and rolling window stats from keypoint data per animal ID (A or B)
6. Train Classifier
    Train Random Forest classifier to detect grooming classes per frame: A_gr_B, B_gr_A, no_grooming
7. Postprocessing
    Classifer predicions are noisy and require postprocessing steps before extracting bout information
    i.e. remove short bouts, merge bouts seperated by short gaps, smooth direction predictions
    Then, extract final bouts and calculate predicted bout metrics compared to annotated bouts (bout count, duration, and direction accuracy)

Once the models have been trained and the pipeline has been established
Run the pipeline on full video dataset

8. YOLO & Bytetrack (step 2)
9. Preprocess DLC data and assign A and B (step 3)
10. Quality control
    Filter out low quality videos where classification is likely to be unreliable
11. Feature Extraction (step 5)
12. Classification and postprocessing (step 7)
    -> outputs csv with predicted bouts for each video

# 1. Multi-Animal Object Classification - Training YOLO Model

1. Load h5 files
2. Extract Keypoints
3. Save as CSV
4. Generate bounding boxes from keypoints
5. Filter implausible bounding boxes - keypoint likelihood, head coherence, spine linearity, keypoint spread
7. Save passing frames and bbox coordinates
8. Split into train and val (80/20)
9. Train YOLO model on bounding boxes
10. Evaluate best model

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import random
import cv2
import yaml
from ultralytics import YOLO

## Convert h5 files to csv

In [ ]:
# Settings
h5_folder = r"DATA_DIR\training_h5"
output_folder = r"DATA_DIR\YOLO_training\training_keypoints"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def extract_keypoints(h5_path):
    """
    Extract raw DLC keypoints from h5 for all individuals
    Returns df with columns:
        frame, identity_{bp}_x, identity_{bp}_y, identity_{bp}_l
    """
    data = pd.read_hdf(h5_path)
    
    # Flatten multiindex
    data.columns = ['_'.join(col).strip() for col in data.columns.values]
    data.index.name = 'frame'
    data = data.reset_index()
    
    # Rename columns to strip scorer prefix
    new_cols = {'frame': 'frame'}
    for col in data.columns:
        if col == 'frame':
            continue
        for individual in ['animal0', 'animal1', 'animal2']:
            if f'_{individual}_' in col:
                # Extract everything from individual name onwards
                suffix = col[col.index(f'_{individual}_') + 1:]
                # Change '_likelihood' to '_l'
                suffix = suffix.replace('_likelihood', '_l')
                new_cols[col] = suffix
                break
    
    data = data.rename(columns=new_cols)
    return data

In [ ]:
# Load h5 files, extract keypoints, and save as csv
h5_files = glob.glob(os.path.join(h5_folder, "*.h5"))
print(f"Found {len(h5_files)} h5 files")

for h5_path in sorted(h5_files):
    video_name = os.path.basename(h5_path).split("_superanimal") [0]
    out_path = os.path.join(output_folder, video_name + " keypoints.csv")

    print(f"Processing {video_name}")

    try:
        df = extract_keypoints(h5_path)
        df.to_csv(out_path, index=False)
        print(f"Saved: {video_name + " keypoints.csv"}")

    except Exception as e:
        print(f"Error: {e}")
        continue

## Generate Bounding Boxes

In [ ]:
csv_folder = r"DATA_DIR\YOLO_training\training_keypoints"
video_folder = r"DATA_DIR\training_videos" 
output_folder = r"DATA_DIR\YOLO_training"

# Bounding Box Settings
img_w, img_h = 960, 540
box_padding = 40
min_box_area = 4000 # filters out small detections (juveniles)
min_box_quality = 0.1
frame_stride = 5
train_split = 0.8

all_parts = [
    'nose', 'mouth_end_right', 'mouth_end_left',
    'right_earbase', 'left_earbase', 'neck_base',
    'back_middle', 'tail_base', 'front_left_paw', 'front_right_paw'
]

# Output folders
os.makedirs(f"{output_folder}/images/train", exist_ok=True)
os.makedirs(f"{output_folder}/images/val", exist_ok=True)
os.makedirs(f"{output_folder}/labels/train", exist_ok=True)
os.makedirs(f"{output_folder}/labels/val", exist_ok=True)

In [ ]:
def keypoints_to_bbox(row, identity, parts=all_parts):
    """
    Derive bounding box from keypoints for one identity in one frame
    Uses likelihood > 0 as the threshold
    Returns (x1, y1, x2, y2) or None if insufficient points
    """
    xs, ys = [], []
    for bp in parts:
        x_col = f'{identity}_{bp}_x'
        y_col = f'{identity}_{bp}_y'
        l_col = f'{identity}_{bp}_l'
        if x_col not in row.index:
            continue
        l = row[l_col]
        if pd.isna(l) or l <= 0:
            continue
        x = row[x_col]
        y = row[y_col]
        if pd.isna(x) or pd.isna(y):
            continue
        xs.append(x)
        ys.append(y)

    if len(xs) < 3:  # need at least 4 points for a reliable box
        return None

    x1 = max(0, int(min(xs) - box_padding))
    y1 = max(0, int(min(ys) - box_padding))
    x2 = min(img_w, int(max(xs) + box_padding))
    y2 = min(img_h, int(max(ys) + box_padding))

    if (x2 - x1) * (y2 - y1) < min_box_area:
        return None

    return x1, y1, x2, y2

def to_yolo(x1, y1, x2, y2):
    """
    Convert pixel bbox to YOLO normalised format
    """
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

def assess_box_quality(row, identity, parts=all_parts):
    """
    Checks:
    1. Enough confident keypoints
    2. Head cluster is coherent (tight group of head points)
    3. Spine is linear (neck -> back -> tail should be roughly collinear)
    3. Box is anatomically coherent (aspect ratio, size)
    4. Keypoints aren't too spaced out

    Returns (bbox, quality_score, reason_failed) or (None, 0, reason)
    """

    head_parts  = ['nose', 'mouth_end_right', 'mouth_end_left', 'right_earbase', 'left_earbase']
    spine_parts = ['neck_base', 'back_middle', 'tail_base']

    # Check 1: confident keypoints
    xs, ys, ls = [], [], []
    for bp in parts:
        x = row.get(f'{identity}_{bp}_x', np.nan)
        y = row.get(f'{identity}_{bp}_y', np.nan)
        l = row.get(f'{identity}_{bp}_l', 0)
        if pd.isna(x) or pd.isna(y):
            continue
        xs.append(x)
        ys.append(y)
        ls.append(float(l) if not pd.isna(l) else 0.0)

    if len(xs) < 4:
        return None, 0.0, 'too_few_points'

    # Check 2: head coherence
    # Head points should form a tight cluster
    # If they are scattered they are likely split across animals
    head_xs, head_ys = [], []
    for bp in head_parts:
        l = row.get(f'{identity}_{bp}_l', 0)
        x = row.get(f'{identity}_{bp}_x', np.nan)
        y = row.get(f'{identity}_{bp}_y', np.nan)
        if not pd.isna(l) and l > 0 and not pd.isna(x):
            head_xs.append(x)
            head_ys.append(y)

    head_coherence = 1.0
    if len(head_xs) >= 3:
        head_spread = np.sqrt(np.var(head_xs) + np.var(head_ys))
        if head_spread > 120:  # pixels, too spread = bad detection
            return None, 0.0, 'head_incoherent'
        head_coherence = np.exp(-head_spread / 60)
    else:
        head_coherence = 0.3  # penalise but don't fail if few head points

    # Check 3: Spine linearity
    spine_pts = []
    for bp in spine_parts:
        l = row.get(f'{identity}_{bp}_l', 0)
        x = row.get(f'{identity}_{bp}_x', np.nan)
        y = row.get(f'{identity}_{bp}_y', np.nan)
        if not pd.isna(l) and l > 0 and not pd.isna(x):
            spine_pts.append([x, y])

    spine_score = 1.0
    if len(spine_pts) == 3:
        p1, p2, p3 = [np.array(p) for p in spine_pts]
        area = abs((p2-p1)[0]*(p3-p1)[1] - (p3-p1)[0]*(p2-p1)[1]) / 2
        length = max(np.linalg.norm(p3 - p1), 1e-6)
        deviation = area / length
        if deviation > 60:  # too bent = cross-animal contamination
            return None, 0.0, 'spine_not_linear'
        spine_score = np.exp(-deviation / 30)

    # Check 4: Box aspect ratio and size
    x1 = max(0, int(min(xs) - box_padding))
    y1 = max(0, int(min(ys) - box_padding))
    x2 = min(img_w, int(max(xs) + box_padding))
    y2 = min(img_h, int(max(ys) + box_padding))

    box_w  = x2 - x1
    box_h  = y2 - y1
    area   = box_w * box_h
    aspect = box_w / max(box_h, 1)

    if area < min_box_area:
        return None, 0.0, 'box_too_small'

    if area > img_w * img_h * 0.5:
        return None, 0.0, 'box_too_large' 

    # Very tall thin boxes or very wide flat boxes are implausible
    if aspect < 0.3 or aspect > 3.0:
        return None, 0.0, 'bad_aspect_ratio'

    # Check 5: Keypoint spread
    kp_spread = np.sqrt(np.var(xs) + np.var(ys))
    expected_spread = np.sqrt(box_w**2 + box_h**2) / 3
    if kp_spread > expected_spread * 2:
        return None, 0.0, 'keypoints_too_scattered'


    # Combined quality score
    mean_likelihood = np.mean(ls)
    n_points_score  = min(len(xs) / 8, 1.0)  # max at 8+ points

    quality = (
        0.40 * head_coherence   +
        0.25 * spine_score      +
        0.20 * mean_likelihood  +
        0.15 * n_points_score
    )

    return (x1, y1, x2, y2), quality, 'ok'

## Filter bounding boxes

In [ ]:
random.seed(42)

csv_files = glob.glob(os.path.join(csv_folder, "*keypoints.csv"))
print(f"Found {len(csv_files)} CSVs")

total_saved = 0
individuals = ['animal0', 'animal1', 'animal2']
for csv_path in csv_files:
    csv_name = os.path.basename(csv_path)
    video_name = csv_name.replace(" keypoints.csv", "")
    video_path = os.path.join(video_folder, video_name + ".mp4")

    if not os.path.exists(video_path):
        print(f"No video for {csv_name}")
        continue

    df = pd.read_csv(csv_path, index_col=0)
    cap = cv2.VideoCapture(video_path)
    n_frames = len(df)

    qualifying = []
    fail_reasons = {}

    for i in range(0, n_frames, frame_stride):
        row = df.iloc [i]

        # Get bounding boxes for all individuals
        boxes = {}
        quals = {}

        for ind in individuals:
            bbox, qual, reason = assess_box_quality(row, ind)
            if bbox is not None:
                boxes[ind] = bbox
                quals[ind] = qual
            else:
                fail_reasons[reason] = fail_reasons.get(reason, 0) + 1

        # Need at least 2 individuals with valid boxes
        if len(boxes) < 2:
            fail_reasons['insufficient_individuals'] = \
                fail_reasons.get('insufficient_individuals', 0) + 1
            continue

        # Select the two highest quality individuals
        top_two = sorted(quals, key=quals.get, reverse=True)[:2]
        min_qual = min(quals[ind] for ind in top_two)

        if min_qual < min_box_quality:
            fail_reasons['low_quality'] = \
                fail_reasons.get('low_quality', 0) + 1
            continue

        bbox_1 = boxes[top_two[0]]
        bbox_2 = boxes[top_two[1]]
        qualifying.append((i, bbox_1, bbox_2))

    if not qualifying:
        print(f"No qualifying frames for {video_name} | {fail_reasons}")
        continue

    print(f"{video_name}: {len(qualifying)} qualifying | {fail_reasons}")

    frame_idx = 0
    qual_idx  = 0
    saved_this_video = 0

    while cap.isOpened() and qual_idx < len(qualifying):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx == qualifying[qual_idx][0]:
            target_i, bbox_1, bbox_2 = qualifying[qual_idx]

            split = 'train' if random.random() < train_split else 'val'
            safe_name = video_name.replace(' ', '_')
            fname = f"{safe_name}_f{frame_idx:05d}"

            img_path = f"{output_folder}/images/{split}/{fname}.jpg"
            cv2.imwrite(img_path, frame)

            lbl_path = f"{output_folder}/labels/{split}/{fname}.txt"
            with open(lbl_path, 'w') as f:
                f.write(to_yolo(*bbox_1) + '\n')
                f.write(to_yolo(*bbox_2) + '\n')

            saved_this_video += 1
            total_saved += 1
            qual_idx += 1

        frame_idx += 1

    cap.release()
    print(f"Saved {saved_this_video} annotated frames")

print(f"Total frames saved: {total_saved}")

In [ ]:
# Check bounding box images
# Random sample 50 images

check_files = random.sample(
    glob.glob(f"{output_folder}/images/train/*.jpg"), 50
)

for img_path in check_files:
    img = cv2.imread(img_path)
    lbl_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')

    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            cx, cy, w, h = map(float, parts[1:])
            x1 = int((cx - w/2) * img_w)
            y1 = int((cy - h/2) * img_h)
            x2 = int((cx + w/2) * img_w)
            y2 = int((cy + h/2) * img_h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Save bbox images 
    fname = os.path.basename(img_path).replace('.jpg', '_check.jpg')
    os.makedirs(f"{output_folder}/images/check", exist_ok=True)
    out_path = os.path.join(output_folder, 'images/check', fname)
    cv2.imwrite(out_path, img)

print(f"Saved bbox images, check: {output_folder}\\images\\check")

## Train YOLO model

In [ ]:
dataset_config = {
    'path': output_folder,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1, # 1 class (monkey)
    'names': ['monkey']
}

config_path = os.path.join(output_folder, 'monkey_dataset.yaml')
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"Dataset config saved: {config_path}")
print(f"Train images: {len(glob.glob(output_folder + '/images/train/*.jpg'))}")
print(f"Val images: {len(glob.glob(output_folder + '/images/val/*.jpg'))}")

In [ ]:
# Start from pretrained YOLOv8n (nano - smallest and fastest)
model = YOLO('yolov8n.pt')

results = model.train(
    data = config_path,
    epochs = 50,
    imgsz = 640,
    batch = 8,
    patience = 15, # stop early if val loss doesn't improve for 15 epochs
    device = 0, # use GPU
    project = output_folder,
    name = 'yolo_monkey',
    exist_ok= True,

    # Augmentation - helps model generalise to camera shake, lighting variation, occlusion
    hsv_h = 0.015, # hue variation
    hsv_s = 0.7, # saturation variation
    hsv_v = 0.4, # brightness variation
    fliplr = 0.5, # horizontal flip
    mosaic = 0.5, # mosaic augmentation
    scale = 0.5, # random scaling - handles distance variation
    translate = 0.1, # random translation
    degrees = 10.0, # slight rotation - handles camera tilt
)

print(f"Best model saved: {results.save_dir}/weights/best.pt")

In [ ]:
# Evaluate best model
best_model = YOLO(f"{output_folder}/yolo_monkey/weights/best.pt")

metrics = best_model.val(data=config_path)

print(f"mAP50: {metrics.box.map50:.3f}")  # main metric (good > 0.7)
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

In [ ]:
# Visually validate
# Run model on video to see how bboxes look

test_video = r"DATA_DIR\training_videos\*.mp4"
output_path = r"DATA_DIR\YOLO_training\*.mp4"

cap = cv2.VideoCapture(test_video)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

best_model = YOLO(f"{output_folder}/yolo_monkey/weights/best.pt")

results = best_model.predict(
    source = test_video,
    conf = 0.3,
    iou = 0.5,
    stream = True,
    verbose = False
)

total_detections = 0
for frame_idx, r in enumerate(results):
    frame = r.orig_img.copy()
    n_detected = len(r.boxes)
    total_detections += n_detected

    for box in r.boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        conf = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{conf:.2f}", (x1, max(y1 - 8, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(frame,
                f"Frame {frame_idx:04d} | Detections: {n_detected}",
                (8, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    out.write(frame)

out.release()
print(f"Saved: {output_path}")

# 2. Tracking - YOLO & Bytetrack

1. Run best yolo model on training videos
2. Run Bytetrack to connect bboxes into tracks
3. Stitch tracks together that are spatially close
4. Assign tracks A and B identities by spatial position
5. Interpolate bboxes
6. Apply hysteresis to suppress brief switches
7. Save A and B bbox coordinates to csv

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import supervision as sv
from ultralytics import YOLO
from itertools import combinations

warnings.filterwarnings('ignore')

In [ ]:
# Settings
yolo_path = r"DATA_DIR\YOLO_training\yolo_monkey\weights\best.pt"
video_folder = r"DATA_DIR\training_videos" 
bbox_output = r"DATA_DIR\training_box_tracks"
os.makedirs(bbox_output, exist_ok=True)

# YOLO & Bytetrack settings
img_w, img_h = 960, 540 # video resolution
fps = 25 # frames per second
confidence_threshold = 0.20
iou_threshold = 0.40
min_adult_box_area = 4000 # filter out babies
track_activation_threshold = 0.25
lost_track_buffer = 30 
minimum_matching_threshold = 0.8
momentum_window = 5 # frames averaged for position smoothing
hysteresis_window = 8 # frames before accepting identity swap
swap_margin = 20 # pixel margin before accepting swap
min_track_frames = 10 # minimum frames a track must appear in to be considered as grooming pair
min_overlap_frames = 10 # minimum frames each tracked pair should appear in together to be considered a grooming pair

In [ ]:
def run_yolo_bytetrack(video_path, yolo_model):
    """
    YOLO + ByteTrack: Assign a consistent ID to each detected animal across frames
    and record the coordinates for each tracked animal

    YOLO: object detection
    ByteTrack: multi-animal tracking

    Returns:
        frame_boxes: list of {track_id: (x1,y1,x2,y2)} per frame
        track_counts: counts how many frames each object appears
        track_centroids: {track_id: {frame_idx: (cx,cy)}}
    """

    # Initialise ByteTrack
    tracker = sv.ByteTrack(
        frame_rate = fps,
        track_activation_threshold = track_activation_threshold,
        lost_track_buffer = lost_track_buffer,
        minimum_matching_threshold = minimum_matching_threshold
    )

    frame_boxes = [] # list of {track_id: (x1,y1,x2,y2)} per frame
    track_counts = {} # number of tracks detected in total
    track_centroids = {} # {track_id: {frame_idx: (cx,cy)}}

    # Run yolo model
    results = yolo_model.predict(
        source = video_path,
        conf = confidence_threshold,
        iou = iou_threshold,
        stream = True,
        verbose = False
    ) 

    for frame_idx, result in enumerate(results):
        detections = sv.Detections.from_ultralytics(result) # converts Ultralytics result into supervision detection object

        if len(detections) > 0:
            areas = ((detections.xyxy[:, 2] - detections.xyxy[:, 0]) *
                     (detections.xyxy[:, 3] - detections.xyxy[:, 1])) 
            
                    # detections.xyxy = (N, 4) - bounding box coordinates 
                    # Area of each box = (x2 - x1)*(y2 - y1) 
            
            detections = detections[areas >= min_adult_box_area] # filter out juveline bboxes
        
        tracked = tracker.update_with_detections(detections)
        f_boxes = {} # stores every tracked identity per frame

        if len(tracked) > 0 and tracked.tracker_id is not None:
            for t_idx in range(len(tracked)):
                t_id = int(tracked.tracker_id[t_idx]) # get tracking ID
                box = tuple(float(v) for v in tracked.xyxy[t_idx]) # get coordinates of bbox
                f_boxes[t_id] = box # store bounding box
                track_counts[t_id] = track_counts.get(t_id, 0) + 1 # count number of appearances
                x1, y1, x2, y2 = box # get bbox coordinates
                track_centroids.setdefault(t_id, {})[frame_idx] = (
                    (x1+x2)/2, (y1+y2)/2
                ) # calculate centroid of bbox

        frame_boxes.append(f_boxes)
        
    return frame_boxes, track_counts, track_centroids

def stitch_tracks(track_centroids, track_counts, frame_boxes, max_gap_frames=30,
                  max_stitch_dist=150, max_size_diff=0.4):
    """
    Group short ByteTrack segments into longer chains representing the same animal

    1. Find where each tracks starts and ends
    2. When a track ends, look for a track that starts within max_gap_frames that is spatially close
    3. Link the tracks into a chain

    Parameters
    
    max_gap_frames: maximum frames between end of one segment and
        start of next to consider stitching
    max_stitch_dist: maximum pixel distance between end position of
        one track and start position of next
    max_size_diff: maximum fractional difference in box area to
        consider same animal (filters infant/adult confusion)
    """
    
    ## Build track segment info
    # For each track: first frame, last frame, start pos, end pos, mean area
    track_info = {}
    for t_id, cents in track_centroids.items():
        frames = sorted(cents.keys())
        first_f = frames [0]
        last_f = frames[-1]
        start_pos = np.array(cents[first_f])
        end_pos = np.array(cents[last_f])

        # Mean box area for size comparison
        areas = []
        for fi in frames:
            box = frame_boxes[fi].get(t_id)
            if box:
                areas.append((box[2]-box[0])*(box[3]-box[1]))
        
        mean_area = np.mean(areas) if areas else 0

        track_info[t_id] = {
            'first_frame' : first_f,
            'last_frame' : last_f,
            'start_pos' : start_pos,
            'end_pos' : end_pos,
            'mean_area' : mean_area,
            'n_frames' : track_counts[t_id]
        }
        
    ## Find next best track after t1 ends
    # For each track, find the ebst candidate to continue from it
    successors = {}
    for t1, info1 in track_info.items():
        best_next = None
        best_cost = np.inf

        for t2, info2 in track_info.items():
            if t2 == t1:
                continue

            # t2 must start after t1 ends
            gap = info2['first_frame'] - info1['last_frame']
            if gap < 0 or gap > max_gap_frames:
                continue

            # Spatial distance between end of t1 and start of t2
            dist = np.linalg.norm(info1['end_pos'] - info2['start_pos'])
            if dist > max_stitch_dist:
                continue

            # Size similarity check (to avoid joining with babies)
            if info1['mean_area'] > 0 and info2['mean_area'] > 0:
                size_ratio = abs(info1['mean_area'] - info2['mean_area']) / \
                             max(info1['mean_area'], info2['mean_area'])
                if size_ratio > max_size_diff:
                    continue
            
            # Cost = distance + time gap penalty
            cost = dist + gap * 2
            if cost < best_cost:
                best_cost = cost
                best_next = t2
        
        if best_next is not None:
            successors[t1] = best_next

    ## Build chains
    # Follow successor links to build chains
    assigned = set()
    chains = []

    # Start chains from tracks with no predecessor
    predecessors = set(successors.values())
    chain_starts = [t for t in track_info if t not in predecessors]

    for start in sorted(chain_starts, key=lambda t: track_info[t]['first_frame']):
        if start in assigned:
            continue

        chain = [start]
        assigned.add(start)
        current = start

        while current in successors:
            nxt = successors[current]
            if nxt in assigned:
                break
            chain.append(nxt)
            assigned.add(nxt)
            current = nxt
        
        chains.append(chain)

    # Add any unassigned tracks as their own single chain
    for t in track_info:
        if t not in assigned:
            chains.append([t])
    
    ## Chain stats
    chain_stats = []
    for chain in chains:
        total_frames = sum(track_info[t]['n_frames'] for t in chain)
        first_frame = min(track_info[t]['first_frame'] for t in chain)
        last_frame = max(track_info[t]['last_frame'] for t in chain)

        all_cents = {}
        for t in chain:
            all_cents.update(track_centroids[t])

        mean_cx = np.mean([cx for cx,cy in all_cents.values()])
        mean_cy = np.mean([cy for cx,cy in all_cents.values()])

        chain_stats.append({
            'chain' : chain,
            'total_frames': total_frames,
            'first_frame' : first_frame,
            'last_frame' : last_frame,
            'centroids' : all_cents,
            'mean_cx' : mean_cx,
            'mean_cy' : mean_cy,
            'span' : last_frame - first_frame
        })
    
    chain_stats.sort(key=lambda x: x['total_frames'], reverse=True)

    print(f"Built {len(chains)} chains from {len(track_info)} tracks")
    for i, cs in enumerate(chain_stats[:6]):
        print(f"Chain {i}: segements = {cs['chain']},"
              f"span={cs['first_frame']}-{cs['last_frame']}, "
              f"mean_cx={cs['mean_cx']:.1f}")
        
    return chain_stats

def build_AB_timelines(chain_stats, frame_boxes, n_frames, min_segment_frames=5):
    """
    Build two complete box timelines (A and B) covering the whole video
    by selecting the best available chain segment at each point in time.
    """

    ## Build lookup
    # For each frame, which chains are active and what are their boxes?
    frame_chain_boxes = []
    for i in range(n_frames):
        active = {}
        if i < len(frame_boxes):
            for chain_idx, cs in enumerate(chain_stats):
                for t_id in cs['chain']:
                    if t_id in frame_boxes[i]:
                        active[chain_idx] = frame_boxes[i][t_id]
                        break
        frame_chain_boxes.append(active)

    ## Score every chain pair for each temporal window
    # Divide video into windows and find best pair per window
    window_size = 125
    n_windows = max(1, n_frames // window_size)

    # For each window score pairs by:
    # 1. Co-detection rate (both present at the same time)
    # 2. Median distanc between centroids (closer - more likely grooming pair)

    window_pair_scores =  []
    for w in range (n_windows):
        w_start = w*window_size
        w_end = min((w+1)*window_size, n_frames)
        w_frames = set(range(w_start, w_end))
        
        # Find chains active in current window
        active_chains = set()
        for i in range (w_start, min(w_end, len(frame_chain_boxes))):
            active_chains.update(frame_chain_boxes[i].keys())
        
        if len(active_chains) < 2:
            window_pair_scores.append(None)
            continue
        
        # Score all active pairs in current window
        best_pair = None
        best_score = -np.inf

        for ci1 in active_chains:
            for ci2 in active_chains:
                if ci2 <= ci1:
                    continue

                # Frames where both chains co-detected
                common = [
                    i for i in range(w_start, min(w_end, len(frame_chain_boxes)))
                    if ci1 in frame_chain_boxes[i]
                    and ci2 in frame_chain_boxes[i]
                ]

                if len(common) < min_segment_frames:
                    continue
                
                # Median distance between centroids
                dists = []
                for i in common:
                    b1 = frame_chain_boxes[i][ci1]
                    b2 = frame_chain_boxes[i][ci2]
                    cx1 = (b1[0]+b1[2])/2
                    cy1 = (b1[1]+b1[3])/2
                    cx2 = (b2[0]+b2[2])/2
                    cy2 = (b2[1]+b2[3])/2
                    dists.append(np.sqrt((cx1-cx2)**2 + (cy1-cy2)**2))

                med_dist = np.median(dists)
                codet_rate = len(common) / (w_end - w_start)

                diag  = np.sqrt(960**2 + 540**2)
                distance_score = 1 - (med_dist / diag)
                score = (0.5 * codet_rate + 0.5 * distance_score)

                if score > best_score:
                    best_score = score
                    best_pair = (ci1, ci2)
        
        window_pair_scores.append(best_pair)

    print(f"Window pair assignments: ")
    for w, pair in enumerate(window_pair_scores):
        w_start = w * window_size
        w_end = min((w+1)*window_size, n_frames)
        print(f"Window {w} (frames {w_start}-{w_end}): "
              f"chains {pair}")
        
    ## Build per-frame assignment using window scores
    frame_A_chain = [None] * n_frames
    frame_B_chain = [None] * n_frames

    last_pair = None

    for w in range(n_windows):
        w_start = w * window_size
        w_end = min((w+1)*window_size, n_frames)
        pair = window_pair_scores[w]

        if pair is not None:
            ci1, ci2 = pair
            cs1 = chain_stats[ci1]
            cs2 = chain_stats[ci2]

            # Determine which is A(left) and which is B(right)
            def window_mean_cx(cs, w_start, w_end):
                xs = [
                    (frame_chain_boxes[i][chain_stats.index(cs)][0] +
                    frame_chain_boxes[i][chain_stats.index(cs)][2]) / 2
                    for i in range(w_start, min(w_end,
                                                len(frame_chain_boxes)))
                    if chain_stats.index(cs) in frame_chain_boxes[i]
                ]
                return np.mean(xs) if xs else 480.0

            cx1 = window_mean_cx(cs1, w_start, w_end)
            cx2 = window_mean_cx(cs2, w_start, w_end)

            if last_pair is not None:
                # Try to maintain consistency with previous window
                # Check if chain assignment matched previous
                prev_A_ci, prev_B_ci = last_pair
                if ci1 == prev_A_ci or ci2 == prev_B_ci:
                    # Consistent with previous
                    ci_A = ci1 if ci1 == prev_A_ci else ci2
                    ci_B = ci2 if ci_A == ci1 else ci1
                elif ci1 == prev_B_ci or ci2 == prev_A_ci:
                    # Swapped relative to previous - keep previous order
                    ci_A = ci2 if ci2 == prev_A_ci else ci1
                    ci_B = ci1 if ci_A == ci2 else ci2
                else:
                    # Completely new pair - use spatial x to assign
                    ci_A = ci1 if cx1 <= cx2 else ci2
                    ci_B = ci2 if ci_A == ci1 else ci1
            else:
                # First window with detection - use spatial x
                ci_A = ci1 if cx1 <= cx2 else ci2
                ci_B = ci2 if ci_A == ci1 else ci1

            last_pair = (ci_A, ci_B)
        
        # Assign frames in this window using last known pair
        if last_pair is not None:
            ci_A, ci_B = last_pair
            for i in range(w_start, w_end):
                if i >= len(frame_chain_boxes):
                    break
                if ci_A in frame_chain_boxes[i]:
                    frame_A_chain[i] = ci_A
                if ci_B in frame_chain_boxes[i]:
                    frame_B_chain[i] = ci_B

    frame_A = [None] * n_frames
    frame_B = [None] * n_frames

    for i in range(n_frames):
        if i >= len(frame_chain_boxes):
            break

        ci_A = frame_A_chain[i]
        ci_B = frame_B_chain[i]

        if ci_A is not None and ci_A in frame_chain_boxes[i]:
            frame_A[i] = frame_chain_boxes[i][ci_A]
        if ci_B is not None and ci_B in frame_chain_boxes[i]:
            frame_B[i] = frame_chain_boxes[i][ci_B]

        # If one is missing but other boxes present, assign best guess
        active_cis = set(frame_chain_boxes[i].keys())
        assigned = {ci_A, ci_B} - {None}
        unassigned = active_cis - assigned

        if frame_A[i] is None and unassigned:
            # Pick closest unassigned chain to A's last known position
            if ci_B is not None and ci_B in frame_chain_boxes[i]:
                b_box = frame_chain_boxes[i][ci_B]
                b_cx  = (b_box[0]+b_box[2])/2
                # A should be on opposite side of B
                candidates = sorted(
                    unassigned,
                    key = lambda ci: abs(
                        (frame_chain_boxes[i][ci][0] +
                         frame_chain_boxes[i][ci][2])/2 - b_cx
                    ),
                    reverse=True  # furthest from B = more likely A
                )
                if candidates:
                    frame_A[i] = frame_chain_boxes[i][candidates[0]]

        if frame_B[i] is None and unassigned:
            active_cis = set(frame_chain_boxes[i].keys())
            assigned = {ci_A, ci_B} - {None}
            if frame_A[i] is not None:
                a_box = frame_A[i]
                a_cx = (a_box[0]+a_box[2])/2
                remaining = active_cis - assigned
                candidates = sorted(
                    remaining,
                    key=lambda ci: abs(
                        (frame_chain_boxes[i][ci][0] +
                         frame_chain_boxes[i][ci][2])/2 - a_cx
                    ),
                    reverse=True
                )
                if candidates:
                    frame_B[i] = frame_chain_boxes[i][candidates[0]]

    a_cov = sum(1 for b in frame_A if b is not None)
    b_cov = sum(1 for b in frame_B if b is not None)
    print(f"Final A coverage: {a_cov}/{n_frames} ({a_cov/n_frames*100:.1f}%)")
    print(f"Final B coverage: {b_cov}/{n_frames} ({b_cov/n_frames*100:.1f}%)")

    return frame_A, frame_B

def interpolate_boxes(box_list, n_frames, max_gap=30):
    """
    Linearly interpolate box coordinates across short gaps.
    Gaps longer than max_gap frames are left as None.
    """
    out = list(box_list)
    i = 0
    while i < n_frames:
        if out[i] is None:
            j = i
            while j < n_frames and out[j] is None:
                j += 1
            gap   = j - i
            before = out[i-1] if i > 0 else None
            after = out[j]   if j < n_frames else None

            if before is not None and after is not None and gap <= max_gap:
                for k in range(gap):
                    t = (k+1) / (gap+1)
                    out[i+k] = tuple(before[c]*(1-t) + after[c]*t
                                     for c in range(4))
            i = j
        else:
            i += 1
    return out


def save_box_csv(frame_A, frame_B, n_frames, out_path):
    """
    Save box assignments to CSV.
    Columns: A_x1, A_y1, A_x2, A_y2, B_x1, B_y1, B_x2, B_y2
    NaN where box not assigned
    """
    rows = []
    for i in range(n_frames):
        row = {'frame': i}
        for identity, box in [('A', frame_A[i]), ('B', frame_B[i])]:
            if box is not None:
                row[f'{identity}_x1'] = box[0]
                row[f'{identity}_y1'] = box[1]
                row[f'{identity}_x2'] = box[2]
                row[f'{identity}_y2'] = box[3]
                row[f'{identity}_cx']  = (box[0]+box[2])/2
                row[f'{identity}_cy']  = (box[1]+box[3])/2
            else:
                row[f'{identity}_x1'] = np.nan
                row[f'{identity}_y1'] = np.nan
                row[f'{identity}_x2'] = np.nan
                row[f'{identity}_y2'] = np.nan
                row[f'{identity}_cx']  = np.nan
                row[f'{identity}_cy']  = np.nan
        rows.append(row)

    pd.DataFrame(rows).set_index('frame').to_csv(out_path)

In [ ]:
# Run step 2
yolo_model = YOLO(yolo_path)
videos = glob.glob(os.path.join(video_folder, "*.mp4"))
print(f"{len(videos)} videos\n")

for i, video_path in enumerate(videos, start=1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    out_path = os.path.join(
        bbox_output,
        video_name.replace(" ", "_") + "_boxes.csv"
    )

    if os.path.exists(out_path):
        continue

    print(f"Processing: {video_name} ({i}/{len(videos)})")

    try:
        frame_boxes, track_counts, track_centroids = run_yolo_bytetrack(video_path, yolo_model)
        n_frames = len(frame_boxes)
        print(f"Frames: {n_frames}, Tracks: {len(track_counts)}")

        if len(track_counts) < 2:
            print("Fewer than 2 tracks detected, skipping")
            continue

        chain_stats = stitch_tracks(track_centroids, track_counts, frame_boxes)
        frame_A, frame_B = build_AB_timelines(chain_stats, frame_boxes, n_frames)
        frame_A = interpolate_boxes(frame_A, n_frames)
        frame_B = interpolate_boxes(frame_B, n_frames)

        a_final = sum(1 for b in frame_A if b is not None)
        b_final = sum(1 for b in frame_B if b is not None)
        print(f"A coverage (final): {a_final/n_frames*100:.1f}%")
        print(f"B coverage (final): {b_final/n_frames*100:.1f}%")

        save_box_csv(frame_A, frame_B, n_frames, out_path)
        print(f"Saved: {os.path.basename(out_path)}\n")

    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        traceback.print_exc()
        continue

# 3. Preprocess DLC data and assign (A and B)

1. Load h5 files
2. For each frame assign DLC animals A and B ID based on how well keypoints fit in bboxes
3. Where coverage is low or boxes are missing - best guess for A and B using nearest box centroids
4. Resolve conflicts - where two slots assigned same identity, keep highest mean likelihood
5. Interpolate gaps in x,y coordinates - liklihood 0 for interpolated keypoints
6. Kalman smoothing per ID
7. Save keypoint data as csv

In [ ]:
## Settings

h5_folder = r"DATA_DIR"
box_folder = r"DATA_DIR\training_box_tracks"
clean_output = r"DATA_DIR\training_csv"
os.makedirs(clean_output, exist_ok=True)

all_parts = [
    'nose', 'mouth_end_right', 'mouth_end_left',
    'right_earbase', 'left_earbase', 'neck_base',
    'back_middle', 'tail_base', 'front_left_paw', 'front_right_paw'
]

kalman_process_noise = 5.0
kalman_measurement_noise = 20.0
hysteresis_window = 8
interpolation_max_gap = 10

In [ ]:
class KalmanKeypoint:
    def __init__(self):
        self.initialised = False
        self.x = np.zeros(4)
        self.P = np.eye(4) * 500.0
        self.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], dtype=float)
        self.H = np.array([[1,0,0,0],[0,1,0,0]], dtype=float)
        q = kalman_process_noise
        self.Q = np.array([
            [q/4,0,q/2,0],[0,q/4,0,q/2],
            [q/2,0,q,0],[0,q/2,0,q]
        ])

    def _predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def _update(self, meas, likelihood):
        R = np.eye(2) * (kalman_measurement_noise / (likelihood + 1e-6))
        y = meas - self.H @ self.x
        S = self.H @ self.P @ self.H.T + R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        
    def process(self, x_meas, y_meas, likelihood):
        has = (not np.isnan(x_meas) and not np.isnan(y_meas)
               and float(likelihood) >= 0.05)
        if not self.initialised:
            if has:
                self.x = np.array([x_meas, y_meas, 0., 0.])
                self.initialised = True
            return (float(x_meas) if has else np.nan,
                    float(y_meas) if has else np.nan)
        self._predict()
        if has:
            self._update(np.array([x_meas, y_meas]), float(likelihood))
        return float(self.x[0]), float(self.x[1])

In [ ]:
def assign_keypoints_boxes(df_raw, scorer, animals, box_df):
    """
    For each frame, assign each DLC animal keypoints to A and B boxes
    Assigned by how well the keypoints fit inside the box
    Always assign A and B

    Returns dictionary with A and B keypoint arrays per body part
    """
    n_frames = min(len(df_raw), len(box_df))

    # Pre-extract all keypoint arrays once upfront
    # arrays[animal][bp] = {'x': np.array, 'y': np.array, 'l': np.array}
    arrays = {}
    for animal in animals:
        arrays[animal] = {}
        for bp in all_parts:
            try:
                x = df_raw[scorer][animal][bp]['x'].values[:n_frames].astype(float)
                y = df_raw[scorer][animal][bp]['y'].values[:n_frames].astype(float)
                l = df_raw[scorer][animal][bp]['likelihood'].values[:n_frames].astype(float)
            except KeyError:
                x = np.full(n_frames, np.nan)
                y = np.full(n_frames, np.nan)
                l = np.zeros(n_frames)
            arrays[animal][bp] = {'x': x, 'y': y, 'l': l}

    # Pre-extract box arrays
    box_arrays = {}
    for identity in ['A', 'B']:
        box_arrays[identity] = {
            'x1': box_df[f'{identity}_x1'].values[:n_frames].astype(float),
            'y1': box_df[f'{identity}_y1'].values[:n_frames].astype(float),
            'x2': box_df[f'{identity}_x2'].values[:n_frames].astype(float),
            'y2': box_df[f'{identity}_y2'].values[:n_frames].astype(float),
        }

    # Output arrays
    out = {}
    for identity in ['A', 'B']:
        for bp in all_parts:
            out[f'{identity}_{bp}_x'] = np.full(n_frames, np.nan)
            out[f'{identity}_{bp}_y'] = np.full(n_frames, np.nan)
            out[f'{identity}_{bp}_l'] = np.zeros(n_frames)

    for i in range(n_frames):
        # Get boxes for this frame
        boxes = {}
        for identity in ['A', 'B']:
            x1 = box_arrays[identity]['x1'][i]
            y1 = box_arrays[identity]['y1'][i]
            x2 = box_arrays[identity]['x2'][i]
            y2 = box_arrays[identity]['y2'][i]
            if not any(np.isnan([x1, y1, x2, y2])):
                boxes[identity] = (x1, y1, x2, y2)

        if not boxes:
            continue

        # For each DLC slot, score which box it belongs to
        slot_assignments = {}  # {animal: (identity, coverage, mean_lik)}

        for animal in animals:
            # Collect valid keypoints for this slot this frame
            xs, ys, ls = [], [], []
            for bp in all_parts:
                x = arrays[animal][bp]['x'][i]
                y = arrays[animal][bp]['y'][i]
                l = arrays[animal][bp]['l'][i]
                if not np.isnan(x) and not np.isnan(y) and not np.isnan(l):
                    xs.append(x)
                    ys.append(y)
                    ls.append(l)

            if not xs:
                continue

            if len(boxes) == 1:
                identity = list(boxes.keys())[0]
                slot_assignments[animal] = (
                    identity, 1.0, np.mean(ls)
                )
                continue

            # Count keypoints inside each box
            scores = {}
            for identity, (bx1, by1, bx2, by2) in boxes.items():
                n_in  = sum(1 for x, y in zip(xs, ys)
                            if bx1 <= x <= bx2 and by1 <= y <= by2)
                scores[identity] = n_in

            best_identity = max(scores, key=scores.get)
            best_coverage = scores[best_identity] / len(xs)

            if scores[best_identity] == 0:
                # No keypoints in any box - assign to nearest centroid
                kp_cx = np.mean(xs)
                kp_cy = np.mean(ys)
                best_identity = min(
                    boxes.keys(),
                    key=lambda id_: np.sqrt(
                        (kp_cx-(boxes[id_][0]+boxes[id_][2])/2)**2 +
                        (kp_cy-(boxes[id_][1]+boxes[id_][3])/2)**2
                    )
                )
                best_coverage = 0.0

            slot_assignments[animal] = (
                best_identity, best_coverage, np.mean(ls)
            )

        # Resolve conflicts - two slots assigned same identity
        # keep highest mean likelihood
        identity_best = {}  # {identity: (animal, mean_lik)}
        for animal, (identity, coverage, mean_lik) in \
                slot_assignments.items():
            if identity not in identity_best or \
                    mean_lik > identity_best[identity][1]:
                identity_best[identity] = (animal, mean_lik)

        # Write keypoints to output arrays
        for identity, (animal, _) in identity_best.items():
            for bp in all_parts:
                out[f'{identity}_{bp}_x'][i] = arrays[animal][bp]['x'][i]
                out[f'{identity}_{bp}_y'][i] = arrays[animal][bp]['y'][i]
                out[f'{identity}_{bp}_l'][i] = arrays[animal][bp]['l'][i]

    return out

def interpolate_keypoints(out_df, max_gap=interpolation_max_gap):
    """
    Linear interpolation of keypoint coordinates 
    Only interpolates x and y 
    Likelihood is 0 for interpolated frames 
    Classifier knows these are estimated positions
    """
    for col in out_df.columns:
        if col.endswith('_x') or col.endswith('_y'):
            out_df[col] = out_df[col].interpolate(
                method = 'linear',
                limit = max_gap,
                limit_direction = 'both'
            )
    return out_df

def apply_kalman(out_df, identity):
    """
    Apply per keypoint Kalman smoothing to identity's track
    """
    for bp in all_parts:
        x_col = f'{identity}_{bp}_x'
        y_col = f'{identity}_{bp}_y'
        l_col = f'{identity}_{bp}_l'
        if x_col not in out_df.columns:
            continue
        kf = KalmanKeypoint()
        sx = np.full(len(out_df), np.nan)
        sy = np.full(len(out_df), np.nan)
        for i in range(len(out_df)):
            x = out_df[x_col].iloc[i]
            y = out_df[y_col].iloc[i]
            l = out_df[l_col].iloc[i]
            sx[i], sy[i] = kf.process(
                float(x) if not pd.isna(x) else np.nan,
                float(y) if not pd.isna(y) else np.nan,
                float(l) if not pd.isna(l) else 0.0
            )
        out_df[x_col] = sx
        out_df[y_col] = sy
    return out_df

In [ ]:
# Run Step 3
h5_files = glob.glob(os.path.join(h5_folder, "*.h5"))
print(f"{len(h5_files)} h5 files")

for f in h5_files:
    filename = os.path.basename(f)
    video_name = filename.split("_superanimal")[0]
    box_path = os.path.join(box_folder, video_name.replace(" ", "_") + "_boxes.csv")
    out_path = os.path.join(clean_output, video_name.replace(" ", "_") + "_cleaned.csv")

    if not os.path.exists(box_path):
        print(f"No box CSV for {video_name}")
        continue

    print(f"Processing: {video_name}")

    try:
        df_raw = pd.read_hdf(f)
        scorer = df_raw.columns.get_level_values(0).unique()[0]
        animals = df_raw.columns.get_level_values(1).unique().tolist()
        df_raw = df_raw.replace(-1.0, np.nan)

        box_df = pd.read_csv(box_path, index_col=0)
        n_frames = len(df_raw)

        kp_data = assign_keypoints_boxes(df_raw, scorer, animals, box_df)
        out_df = pd.DataFrame(kp_data)
        out_df.index.name = 'frame'

        out_df = interpolate_keypoints(out_df)
        out_df = apply_kalman(out_df, 'A')
        out_df = apply_kalman(out_df, 'B')

        # Coverage summary
        for identity in ['A', 'B']:
                col = f'{identity}_back_middle_x'
                if col in out_df.columns:
                    cov = out_df[col].notna().mean() * 100
                    print(f"{identity} coverage: {cov:.1f}")

        out_df.to_csv(out_path)
        print(f"Saved: {os.path.basename(out_path)}\n")
    
    except Exception as e:
         import traceback
         print(f"Error: {e}")
         traceback.print_exc()

In [ ]:
# Visually validate keypoint and bbox identities
# Generate videos with assigned bounding boxes and keypoints

video_folder = r"DATA_DIR\training_videos"
box_folder = r"DATA_DIR\training_box_tracks"
csv_folder = r"DATA_DIR\training_csv"
out_folder = r"DATA_DIR\check_videos"
os.makedirs(out_folder, exist_ok=True)

for video_path in sorted(glob.glob(os.path.join(video_folder, "*.mp4"))):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    safe_name = video_name.replace(" ", "_")

    box_path = os.path.join(box_folder, f"{safe_name}_boxes.csv")
    kp_path = os.path.join(csv_folder, f"{safe_name}_cleaned.csv")
    out_path = os.path.join(out_folder, f"{safe_name}_check.mp4")

    if not os.path.exists(box_path):
        print(f"No box CSV for {video_name}, skipping")
        continue
    if not os.path.exists(kp_path):
        print(f"No keypoint CSV for {video_name}, skipping")
        continue
    if os.path.exists(out_path):
        print(f"Already rendered: {video_name}")
        continue

    print(f"\nRendering: {video_name}")

    box_df = pd.read_csv(box_path)
    kp_df = pd.read_csv(kp_path)

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    n_frames = min(total, len(box_df), len(kp_df))

    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    colours = {
        "A": (0, 255, 0), # Green
        "B": (0, 0, 255), # Red
    }

    print(f"Rendering {video_name} ({n_frames} frames)...")

    for frame_idx in range(n_frames):

        ret, frame = cap.read()
        if not ret:
            break

        box_row = box_df.iloc[frame_idx]

        for identity in ["A", "B"]:
            x1 = box_row.get(f"{identity}_x1", np.nan)
            y1 = box_row.get(f"{identity}_y1", np.nan)
            x2 = box_row.get(f"{identity}_x2", np.nan)
            y2 = box_row.get(f"{identity}_y2", np.nan)

            if any(pd.isna(v) for v in [x1, y1, x2, y2]):
                continue

            x1 = int(x1)
            y1 = int(y1)
            x2 = int(x2)
            y2 = int(y2)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                colours[identity],
                2
            )

            cv2.putText(
                frame,
                identity,
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                colours[identity],
                2
            )

        for identity in ["A", "B"]:
            colour = colours[identity]
            for bp in all_parts:
                x_col = f"{identity}_{bp}_x"
                y_col = f"{identity}_{bp}_y"
                l_col = f"{identity}_{bp}_l"

                if x_col not in kp_df.columns:
                    continue

                x = kp_df.at[frame_idx, x_col]
                y = kp_df.at[frame_idx, y_col]
                l = kp_df.at[frame_idx, l_col]

                if pd.isna(x) or pd.isna(y) or pd.isna(l):
                    continue

                if l < 0.3:
                    continue

                cv2.circle(frame, (int(x), int(y)), 4, colour, -1)

        cv2.rectangle(frame, (0, 0), (w, 40), (0, 0, 0), -1)

        n_boxes = 0
        if pd.notna(box_row.get("A_x1", np.nan)):
            n_boxes += 1
        if pd.notna(box_row.get("B_x1", np.nan)):
            n_boxes += 1

        cv2.putText(
            frame,
            f"Frame {frame_idx+1}/{n_frames}   Boxes: {n_boxes}",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            1
        )

        writer.write(frame)

        if (frame_idx + 1) % 250 == 0:
            print(f"  Frame {frame_idx+1}/{n_frames}")

    cap.release()
    writer.release()

    print(f"Saved: {out_path}")

# 4. Parse manually annotated bouts

1. Load manual annotations of training data - containing grooming bout start time, end time, direction, and total number of bouts
2. Parse bouts
3. Create long df (columns: vide, frame, label (A_gr_B, B_gr_A, no_grooming))

In [ ]:
# Import manually encoded labels
annotations = pd.read_csv(r"DATA_DIR\annotations.csv")
csv_folder = r"DATA_DIR\training_csv"
# Removes empty rows where Names is NaN
annotations = annotations.dropna(subset=['Name']).reset_index(drop=True)
annotations = annotations.reset_index(drop=True)

In [ ]:
def parse_all_bouts(annotations, fps=25):
    """
    Creates a dictionary with the information per bout for each video
    - start and end time and total duration 
    - start and end frame
    - label = A gr B, B gr A
    """
    all_video_bouts = {}
    
    for _, row in annotations.iterrows():
        video_name = row['Name']
        bouts = []
        
        for i in range(1, 5):
            start_col = f'bout_{i}_start' # start time
            end_col = f'bout_{i}_end' # end time

            gr_col = f'bout_{i}_gr' # groomer (A or B)
            rec_col = f'bout_{i}_rec' # recipient (A or B)

            if start_col not in row.index or end_col not in row.index:
                continue
            if gr_col not in row.index or rec_col not in row.index:
                continue
            
            start = row[start_col]
            end = row[end_col]

            if pd.isna(start) or pd.isna(end):
                continue

            groomer = row[gr_col]
            recipient = row[rec_col]

            if groomer == 'A' and recipient == 'B':
                label = 'A_gr_B'
            elif groomer == 'B' and recipient == 'A':
                label = 'B_gr_A'
            
            if pd.notna(start) and pd.notna(end):
                bouts.append({
                    'bout_num': i,
                    'start_sec': float(start),
                    'end_sec': float(end),
                    'start_frame': int(float(start) * fps),
                    'end_frame': int(float(end) * fps),
                    'duration_sec': float(end) - float(start),
                    'label': label
                })
        
        if bouts:
            all_video_bouts[video_name] = bouts
    
    return all_video_bouts

In [ ]:
all_video_bouts = parse_all_bouts(annotations)
all_video_bouts

In [ ]:
"""
Create dataframe to assign a label to every frame:
- video name
- frame number
- label = A gr B, B gr A, no_grooming
"""

all_labels = []

for video_name, bouts in all_video_bouts.items():
    
    clean_name = video_name.replace(" ", "_") + "_cleaned.csv" #match names in annotations with csv names
    csv_path = os.path.join(csv_folder, clean_name)

    if not os.path.exists(csv_path):
        print(f"Skipping {video_name}, CSV not found")
        continue

    # load csv
    df = pd.read_csv(csv_path, header=[0,1,2,3], index_col=0)
    n_frames = len(df)

    # create series all 'no_grooming' (default) for the number of frames in the video
    labels = pd.Series('no_grooming', index=range(n_frames))

    # for each frame with grooming, change from 'no_grooming' to 'grooming'
    for bout in bouts:
        start = bout['start_frame']
        end = bout['end_frame']
        bout_label = bout['label']
        labels.iloc[start:end+1] = bout_label

    # Store video name, frame number, and labels together
    video_labels = pd.DataFrame({
        'video' : video_name, 
        'frame' : range(n_frames),
        'label' : labels.values
    })

    # add to dictionary
    all_labels.append(video_labels)


labels_df = pd.concat(all_labels, ignore_index = True)
labels_df.head()

# 5. Feature Extraction

1. Formulaes for distance, velocity and angles
2. Extract distance, velocity, angle, likelihoods, detection flags, and short and long rolling window stats

In [ ]:
# Formulaes 

def calculate_distance(x1, y1, x2, y2):
    """ Calculate Euclidean distance between two points """
    return np.sqrt((x2 - x1)**2 + (y2 -y1)**2)

def calculate_velocity(x, y):
    """ Calculate frame-to-frame velocity of body part """
    return np.sqrt(x.diff()**2 + y.diff()**2)

def calculate_angle(x1, y1, x2, y2):
    """ Calculate the angle from point 1 to point 2 (degrees)
        To determine if one animal is facing the other """
    return np.arctan2(y2 - y1, x1 - x2)*180/np.pi 

In [ ]:
def feature_extraction(df):
    """
    Define and extract grooming features
    1. Inter-animal distances: paw-to-body, centroid distance
    2. Distance likelihoods
    3. Velocities: paw, nose, and body velocity
    5. Angles/orientation: facing direction and body angle
    6. Detection likelihoods per ID per frame
    7. Detection flags: 0:interpolated, 1:true DLC detection
    8. Short and long rolling windows: mean and std for each feature

    df: keypoint csv with assigned A and B IDs
        columns: A_{bp}_{x/y/l}, B_{bp}_{x/y/l},

    Returns feature dataframe: each row is one frame, each column is one feature
    """

    # A body parts
    A_nose_x = df['A_nose_x']
    A_nose_y = df['A_nose_y']
    A_nose_l = df['A_nose_l']

    A_neck_x = df['A_neck_base_x']
    A_neck_y = df['A_neck_base_y']
    A_neck_l = df['A_neck_base_l']

    A_back_x = df['A_back_middle_x']
    A_back_y = df['A_back_middle_y']
    A_back_l = df['A_back_middle_l']

    A_tail_x = df['A_tail_base_x']
    A_tail_y = df['A_tail_base_y']
    A_tail_l = df['A_tail_base_l']

    A_lpaw_x = df['A_front_left_paw_x']
    A_lpaw_y = df['A_front_left_paw_y']
    A_lpaw_l = df['A_front_left_paw_l']

    A_rpaw_x = df['A_front_right_paw_x']
    A_rpaw_y = df['A_front_right_paw_y']
    A_rpaw_l = df['A_front_right_paw_l']

    # B body parts
    B_nose_x = df['B_nose_x']
    B_nose_y = df['B_nose_y']
    B_nose_l = df['B_nose_l']

    B_neck_x = df['B_neck_base_x']
    B_neck_y = df['B_neck_base_y']
    B_neck_l = df['B_neck_base_l']

    B_back_x = df['B_back_middle_x']
    B_back_y = df['B_back_middle_y']
    B_back_l = df['B_back_middle_l']

    B_tail_x = df['B_tail_base_x']
    B_tail_y = df['B_tail_base_y']
    B_tail_l = df['B_tail_base_l']

    B_lpaw_x = df['B_front_left_paw_x']
    B_lpaw_y = df['B_front_left_paw_y']
    B_lpaw_l = df['B_front_left_paw_l']

    B_rpaw_x = df['B_front_right_paw_x']
    B_rpaw_y = df['B_front_right_paw_y']
    B_rpaw_l = df['B_front_right_paw_l']

    features = pd.DataFrame(index=df.index)


    ## Inter-animal distances

    # Paw-to-body distances for A and B
    for paw, paw_x, paw_y in [('lpaw', A_lpaw_x, A_lpaw_y),
                              ('rpaw', A_rpaw_x, A_rpaw_y)]:
        for target, tgt_x, tgt_y in [('nose', B_nose_x, B_nose_y),
                                     ('neck', B_neck_x, B_neck_y),
                                     ('back', B_back_x, B_back_y),
                                     ('tail', B_tail_x, B_tail_y)]:
            features[f'A_{paw}_to_B_{target}'] = calculate_distance(paw_x, paw_y, tgt_x, tgt_y)

    for paw, paw_x, paw_y in [('lpaw', B_lpaw_x, B_lpaw_y),
                              ('rpaw', B_rpaw_x, B_rpaw_y)]:
        for target, tgt_x, tgt_y in [('nose', A_nose_x, A_nose_y),
                                     ('neck', A_neck_x, A_neck_y),
                                     ('back', A_back_x, A_back_y),
                                     ('tail', A_tail_x, A_tail_y)]:
            features[f'B_{paw}_to_A_{target}'] = calculate_distance(paw_x, paw_y, tgt_x, tgt_y)

    # Distance between A and B centroids (middle back)
    features['centroid_distance'] = calculate_distance(A_back_x, A_back_y, B_back_x, B_back_y)

    # Minimum paw-to-body distance per ID, per frame
    A_paw_cols = [c for c in features.columns if c.startswith('A_') and 'paw_to_B' in c]
    B_paw_cols = [c for c in features.columns if c.startswith('B_') and 'paw_to_A' in c]

    features['min_A_paw_to_B_body'] = features[A_paw_cols].min(axis=1)
    features['min_B_paw_to_A_body'] = features[B_paw_cols].min(axis=1)

    # Overall minimum between A and B
    features['min_paw_to_body'] = np.minimum(
        features['min_A_paw_to_B_body'],
        features['min_B_paw_to_A_body']
    )

    # Distance likelihoods, using DLC likelihood, or 0 for interpolated
    # Product of paw and body likelihoods
    likelihoods = {
        'A': {
            'lpaw': A_lpaw_l, 'rpaw': A_rpaw_l,
            'nose': A_nose_l, 'neck': A_neck_l,
            'back': A_back_l, 'tail': A_tail_l,
        },
        'B': {
            'lpaw': B_lpaw_l, 'rpaw': B_rpaw_l,
            'nose': B_nose_l, 'neck': B_neck_l,
            'back': B_back_l, 'tail': B_tail_l,
        }
    }

    # Paw-to-body likelihoods
    for gr, rec in [('A', 'B'), ('B', 'A')]:
        for paw in ['lpaw', 'rpaw']:
            for target in ['nose', 'neck', 'back', 'tail']:
                features[f'{gr}_{paw}_{rec}_{target}_conf'] = (
                    likelihoods[gr][paw] * likelihoods[rec][target]
                )

    # Centroid likelihood
    features['centroid_distance_conf'] = A_back_l * B_back_l


    ## Velocity: paw, body (back), and nose

    features['A_lpaw_velocity'] = calculate_velocity(A_lpaw_x, A_lpaw_y)
    features['A_rpaw_velocity'] = calculate_velocity(A_rpaw_x, A_rpaw_y)
    features['B_lpaw_velocity'] = calculate_velocity(B_lpaw_x, B_lpaw_y)
    features['B_rpaw_velocity'] = calculate_velocity(B_rpaw_x, B_rpaw_y)

    features['A_body_velocity'] = calculate_velocity(A_back_x, A_back_y)
    features['B_body_velocity'] = calculate_velocity(B_back_x, B_back_y)

    features['A_nose_velocity'] = calculate_velocity(A_nose_x, A_nose_y)
    features['B_nose_velocity'] = calculate_velocity(B_nose_x, B_nose_y)

    features['A_mean_paw_velocity'] = (features['A_lpaw_velocity'] + features['A_rpaw_velocity']) / 2
    features['B_mean_paw_velocity'] = (features['B_lpaw_velocity'] + features['B_rpaw_velocity']) / 2

    # Relative body velocity
    features['relative_body_velocity'] = calculate_velocity(
        A_back_x - B_back_x,
        A_back_y - B_back_y
    )


    ## Angles/Orientation

    # A facing B, B facing A
    features['A_facing_B'] = calculate_angle(A_nose_x, A_nose_y, B_nose_x, B_nose_y)
    features['B_facing_A'] = calculate_angle(B_nose_x, B_nose_y, A_nose_x, A_nose_y)

    # Body angle
    features['A_body_angle'] = calculate_angle(A_tail_x, A_tail_y, A_nose_x, A_nose_y)
    features['B_body_angle'] = calculate_angle(B_tail_x, B_tail_y, B_nose_x, B_nose_y)

    # Absolute nody angle difference between individuals
    features['body_angle_diff'] = np.abs(features['A_body_angle'] - features['B_body_angle'])


    ## Detection likelihoods

    # Mean of 5 keypoints per individual
    features['A_mean_likelihood'] = (A_nose_l + A_neck_l + A_lpaw_l + A_rpaw_l + A_back_l) / 5
    features['B_mean_likelihood'] = (B_nose_l + B_neck_l + B_lpaw_l + B_rpaw_l + B_back_l) / 5

    # Likelihood ratio
    features['likelihood_ratio'] = (features['A_mean_likelihood'] / (features['B_mean_likelihood'] + 1e-6))



    ## Detection flags - 0: interpolated, 1: DLC detection
    A_detected = ((A_nose_l > 0) & (A_back_l > 0)).astype(int)
    B_detected = ((B_nose_l > 0) & (B_back_l > 0)).astype(int)

    features['A_detected'] = A_detected
    features['B_detected'] = B_detected
    features['both_detected'] = (A_detected & B_detected).astype(int)
    features['neither_detected'] = ((A_detected == 0) & (B_detected == 0)).astype(int)


    ## Rolling window statistics
    # std not included for all features as it wasn't informative on previous runs

    window_short = 5 # 5 frames
    window_long  = 25 # 25 frames

    distance_roll_features = [
        'min_paw_to_body', 'min_A_paw_to_B_body',
        'min_B_paw_to_A_body', 'centroid_distance'
    ]

    for feat in distance_roll_features:
        features[f'{feat}_mean_{window_short}'] = (features[feat].rolling(window_short, center=True).mean())
        features[f'{feat}_std_{window_short}'] = (features[feat].rolling(window_short, center=True).std())
        features[f'{feat}_mean_{window_long}'] = (features[feat].rolling(window_long, center=True).mean())
        features[f'{feat}_std_{window_long}'] = (features[feat].rolling(window_long, center=True).std())

    velocity_roll_features = [
        'A_mean_paw_velocity', 'B_mean_paw_velocity',
        'A_body_velocity', 'B_body_velocity', 'relative_body_velocity'
    ]

    for feat in velocity_roll_features:
        features[f'{feat}_mean_{window_short}'] = (features[feat].rolling(window_short, center=True).mean())
        features[f'{feat}_mean_{window_long}'] = (features[feat].rolling(window_long, center=True).mean())
        features[f'{feat}_std_{window_short}'] = (features[feat].rolling(window_short, center=True).std())

    angle_roll_features = ['body_angle_diff']

    for feat in angle_roll_features:
        features[f'{feat}_mean_{window_short}'] = (features[feat].rolling(window_short, center=True).mean())
        features[f'{feat}_mean_{window_long}'] = (features[feat].rolling(window_long, center=True).mean())

    detection_roll_features = ['A_detected', 'B_detected', 'both_detected']

    for feat in detection_roll_features:
        features[f'{feat}_mean_{window_short}'] = (features[feat].rolling(window_short, center=True).mean())
        features[f'{feat}_mean_{window_long}'] = (features[feat].rolling(window_long, center=True).mean())

    likelihood_roll_features = [
        'A_mean_likelihood', 'B_mean_likelihood', 'likelihood_ratio',
    ]

    for feat in likelihood_roll_features:
        features[f'{feat}_mean_{window_short}'] = (features[feat].rolling(window_short, center=True).mean())
        features[f'{feat}_mean_{window_long}'] = (features[feat].rolling(window_long, center=True).mean())
        features[f'{feat}_std_{window_short}'] = (features[feat].rolling(window_short, center=True).std())

    return features

In [ ]:
# Run step 5

all_features = []

for video_name in labels_df['video'].unique():
    
    clean_name = video_name.replace(" ", "_") + "_cleaned.csv"
    csv_path = os.path.join(csv_folder, clean_name)
    
    if not os.path.exists(csv_path):
        print(f"Skipping {video_name} — CSV not found")
        continue
    
    print(f"Processing: {video_name}")
    
    df = pd.read_csv(csv_path, index_col=0)
    
    # Extract features
    features = feature_extraction(df).reset_index(drop=True)

    # Add metadata
    features["video"] = video_name
    features["frame"] = features.index

    all_features.append(features)

features_df = pd.concat(all_features, ignore_index=True)
print(f"Features shape: {features_df.shape}")

# 6. Train Classifier

1. Prepare data for classifcation (assign X and y)
2. Train model using Group K-fold cross validation
3. Generate classification report and confusion matrix
4. Fit model
5. Generate RF feature importances

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Prepare data for classification
training_df = pd.merge(
    labels_df,
    features_df,
    on = ['video', 'frame'],
    how = 'inner'
)

feature_cols = [col for col in training_df.columns if col not in ['label', 'video', 'frame']]

X = training_df[feature_cols].copy()
y = training_df['label'].copy()
group = training_df['video'].copy() # For Group K-fold

print(f"X shape: {X.shape}")
print(f"Label distribution:")
print(y.value_counts())

In [ ]:
# Initialise classifier
# Use a pipeline so imputation only sees training data in each fold (prevents leakage)
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=1000,
        min_samples_leaf=20,
        class_weight='balanced',
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# Run classifier
# Group K fold to shuffle labels but keep frames from the same video in train or test sets
cv = GroupKFold(n_splits=min(5, len(group.unique())))
y_proba_cv = cross_val_predict(pipeline, X, y, cv=cv, groups=group, method='predict_proba', n_jobs=-1)

In [ ]:
# Classification report and confusion matrix
classes = np.array(sorted(y.unique()))
y_pred_cv = classes[y_proba_cv.argmax(axis=1)]
print(classification_report(y, y_pred_cv, target_names=classes))

%matplotlib inline
cm = confusion_matrix(y, y_pred_cv, labels = classes)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap='Blues', ax=ax)
plt.title('Cross-validation confusion matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Fit classifier model
pipeline.fit(X, y)

In [ ]:
# Top 20 feature importances
rf = pipeline.named_steps['rf']

# Feature importance dataframe
feature_importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

top20 = feature_importance.head(20)

plt.figure(figsize=(8, 8))
plt.barh(top20["Feature"][::-1], top20["Importance"][::-1])
plt.xlabel("Feature Importance")
plt.title("Top 20 Random Forest Feature Importances")
plt.tight_layout()

# 7. Postprocessing

1. Get rf probabilites for classification classes per frame for each video
2. Clean predictions
    Smooth direction within continuous grooming frames
    Remove bouts shorter than 1 second
    Merge gaps shorter than 0.5 regardless of direction
    Merge gaps shorter than 2 seconds of same-direction bouts
3. Remove isolated short bouts near the end of video
    Shorter than 3 seconds, occuring 30 seconds after previous bout
4. Extract bouts from annotated labels for comparison
5. Calculate bout metrics - comparison between predicted bouts and true bouts
6. Visually compare predicted and true timelines per video

In [ ]:
def get_rf_probabilities(video_name, df, pipeline, feature_cols, fps=25):
    """
    Get per-frame RF probabilities for a single video
    Uses the full pipeline (imputer + RF) so NaN handling is consistent

    Returns df with video, frame, p_grooming, p_A_gr_B, p_B_gr_A, p_no_grooming
    """
    features = feature_extraction(df)
    features = features.reindex(columns=feature_cols)
    
    # Use full pipeline for prediction (handles NaN via imputer)
    proba = pipeline.predict_proba(features)
    classes = list(pipeline.classes_)
    
    p_AgB = proba[:, classes.index('A_gr_B')] \
            if 'A_gr_B' in classes else np.zeros(len(proba))
    p_BgA = proba[:, classes.index('B_gr_A')] \
            if 'B_gr_A' in classes else np.zeros(len(proba))
    p_no = proba[:, classes.index('no_grooming')] \
            if 'no_grooming' in classes else np.zeros(len(proba))
    
    p_grooming = p_AgB + p_BgA
    
    return pd.DataFrame({
        'video' : video_name,
        'frame' : range(len(features)),
        'p_grooming' : p_grooming,
        'p_A_gr_B' : p_AgB,
        'p_B_gr_A' : p_BgA,
        'p_no_grooming' : p_no
    })

In [ ]:
def clean_rf_predictions(rf_output, fps = 25,
                          min_bout_sec = 1, # remove very short bouts
                          noise_gap_sec = 0.5, # merge gaps this short regardless of direction
                          same_dir_gap_sec = 2, # merge same-direction bouts with gaps this short
                          direction_window = 10): # frames to smooth direction within a bout
    """   
    1. Direction smoothing within continuous grooming frames - majority vote over short window
        suppresses brief direction flips 
    2. Extract raw bouts for bout smoothing
    3. Remove bouts shorter than 1 second
    4. Merge short gaps regardless of direction - gaps shorts than 0.5 seconds to handle brief signal dropout
        Then merge gaps between between same-direction bouts with a larger gap threshold (2s)
    5. Remove any leftover short bouts
    """
    min_bout_frames = int(min_bout_sec * fps)
    noise_gap_frames = int(noise_gap_sec * fps)
    same_dir_gap_frames = int(same_dir_gap_sec * fps)

    frames = rf_output.copy()

    # 1. Direction smoothing
    frames['p_A_smooth'] = frames['p_A_gr_B'].rolling(direction_window, center=True, min_periods=1).mean()
    frames['p_B_smooth'] = frames['p_B_gr_A'].rolling(direction_window, center=True, min_periods=1).mean()
    frames['rf_direction'] = np.where(frames['p_A_smooth'] > frames['p_B_smooth'], 'A_gr_B', 'B_gr_A') # direction for window - largest mean prob
    frames['rf_grooming'] = frames['p_grooming'] > 0.6 # only grooming if probability is above 0.6

    # 2. Extract raw bouts
    def extract_bouts(grooming_col, direction_col):
        bouts = []
        in_bout = False
        start = 0
        series_g = frames[grooming_col].values # grooming or no grooming
        series_d = frames[direction_col].values # direction of grooming
        n = len(series_g)

        for i in range(n):
            if series_g[i] and not in_bout:
                in_bout = True
                start = i
            elif not series_g[i] and in_bout:
                in_bout = False
                window = series_d[start:i]
                counts = pd.Series(window).value_counts()
                bouts.append({
                    'start' : start,
                    'end' : i - 1,
                    'n_frames' : i - start,
                    'direction': counts.index[0]
                })
        if in_bout:
            window = series_d[start:]
            counts = pd.Series(window).value_counts()
            bouts.append({
                'start' : start,
                'end' : n - 1,
                'n_frames' : n - start,
                'direction': counts.index[0]
            })
        return bouts

    bouts = extract_bouts('rf_grooming', 'rf_direction')
    print(f"Raw bouts: {len(bouts)}") # raw predicted bouts

    # 3. Remove short bouts
    bouts = [b for b in bouts if b['n_frames'] >= min_bout_frames]

    # 4. Merge short gaps 
    def merge_gaps(bouts, max_gap, same_dir_only=False):
        if len(bouts) <= 1:
            return bouts
        merged = True
        while merged:
            merged = False
            new_bouts = []
            i = 0
            while i < len(bouts):
                if i < len(bouts) - 1:
                    gap = bouts[i+1]['start'] - bouts[i]['end'] - 1
                    same_dir = bouts[i]['direction'] == bouts[i+1]['direction']

                    if gap <= max_gap and (not same_dir_only or same_dir):
                        # Merge, direction assigned to whichever bout is longer
                        longer_dir = (
                            bouts[i]['direction']
                            if bouts[i]['n_frames'] >= bouts[i+1]['n_frames']
                            else bouts[i+1]['direction']
                        )
                        new_bouts.append({
                            'start' : bouts[i]['start'],
                            'end' : bouts[i+1]['end'],
                            'n_frames' : bouts[i+1]['end'] - bouts[i]['start'] + 1,
                            'direction': longer_dir
                        })
                        i += 2
                        merged = True
                    else:
                        new_bouts.append(bouts[i])
                        i += 1
                else:
                    new_bouts.append(bouts[i])
                    i += 1
            bouts = new_bouts
        return bouts

    # First pass: merge tiny noise gaps (any direction)
    bouts = merge_gaps(bouts, noise_gap_frames, same_dir_only=False)

    # Second pass: merge same-direction bouts with moderate gaps
    bouts = merge_gaps(bouts, same_dir_gap_frames, same_dir_only=True)

    if not bouts:
        return pd.DataFrame()

    result = pd.DataFrame([{
        'start_frame' : b['start'],
        'end_frame' : b['end'],
        'n_frames' : b['n_frames'],
        'duration_s' : b['n_frames'] / fps,
        'direction' : b['direction'],
        'mean_p_grooming' : frames['p_grooming'].iloc[b['start']:b['end']+1].mean(),
        'mean_p_A_gr_B' : frames['p_A_gr_B'].iloc[b['start']:b['end']+1].mean(),
        'mean_p_B_gr_A': frames['p_B_gr_A'].iloc[b['start']:b['end']+1].mean(),
        'direction_confidence' : abs(
            frames['p_A_gr_B'].iloc[b['start']:b['end']+1].mean() -
            frames['p_B_gr_A'].iloc[b['start']:b['end']+1].mean()
        )
    } for b in bouts])

    return result

In [ ]:
def finalise_bouts(pred_bouts_video, fps = 25,
                   merge_same_dir_gap_s = 5, # merge same direction bouts with larger threshold
                   end_isolation_gap_s = 30, # gap for potential false positive bouts
                   min_end_bout_s = 3): # not a true bout if shorter than 3 seconds
    """
    Two targeted fixes applied after clean_rf_predictions:

    1. Merge gaps between same-direction bouts 
    2. Remove isolated short bouts near the end of video that are
       separated from the main grooming session by a long gap
       (handles false positives after grooming has ended)
    """
    if len(pred_bouts_video) == 0:
        return pred_bouts_video

    bouts = pred_bouts_video.sort_values('start_frame').copy()
    merge_gap_frames = int(merge_same_dir_gap_s * fps)
    end_gap_frames = int(end_isolation_gap_s  * fps)
    min_end_frames = int(min_end_bout_s * fps)

    # 1: Merge same-direction bouts with small gaps
    changed = True
    while changed:
        changed = False
        rows = bouts.reset_index(drop=True)
        new_rows = []
        i = 0
        while i < len(rows):
            if i < len(rows) - 1:
                curr = rows.iloc[i]
                nxt = rows.iloc[i+1]
                gap = nxt['start_frame'] - curr['end_frame'] - 1
                same_dir = curr['direction'] == nxt['direction']

                if same_dir and gap <= merge_gap_frames:
                    # Merge into one bout
                    merged_frames = nxt['end_frame'] - curr['start_frame'] + 1
                    new_rows.append({
                        **curr.to_dict(),
                        'end_frame' : nxt['end_frame'],
                        'n_frames' : merged_frames,
                        'duration_s' : merged_frames / fps,
                        'mean_p_grooming' : (curr['mean_p_grooming'] * curr['n_frames'] + nxt['mean_p_grooming'] * nxt['n_frames']) /
                                                 merged_frames, 'mean_p_A_gr_B': (curr.get('mean_p_A_gr_B', 0) *
                                                  curr['n_frames'] + nxt.get('mean_p_A_gr_B', 0) *
                                                  nxt['n_frames']) / merged_frames,
                        'mean_p_B_gr_A'       : (curr.get('mean_p_B_gr_A', 0) * curr['n_frames'] +
                                                  nxt.get('mean_p_B_gr_A', 0) * nxt['n_frames']) /
                                                 merged_frames,
                        'direction_confidence': abs(
                            (curr.get('mean_p_A_gr_B', 0) * curr['n_frames'] + nxt.get('mean_p_A_gr_B', 0) *
                             nxt['n_frames']) / merged_frames - (curr.get('mean_p_B_gr_A', 0) *
                             curr['n_frames'] + nxt.get('mean_p_B_gr_A', 0) *
                             nxt['n_frames']) / merged_frames
                        )
                    })
                    i += 2
                    changed = True
                else:
                    new_rows.append(curr.to_dict())
                    i += 1
            else:
                new_rows.append(rows.iloc[i].to_dict())
                i += 1
        bouts = pd.DataFrame(new_rows)

    # 2: Remove isolated short bouts at end of video ─
    if len(bouts) > 1:
        bouts = bouts.sort_values('start_frame').reset_index(drop=True)
        to_drop = []

        for i in range(1, len(bouts)):
            curr = bouts.iloc[i]
            prev = bouts.iloc[i-1]
            gap = curr['start_frame'] - prev['end_frame'] - 1

            # If this bout is far from the previous one
            # and is itself short, it's likely end-of-video noise
            if (gap >= end_gap_frames and
                    curr['n_frames'] < min_end_frames):
                to_drop.append(i)

        if to_drop:
            bouts = bouts.drop(index=to_drop).reset_index(drop=True)

    return bouts

In [ ]:
all_rf_frames = []
all_rf_bouts  = []

for video_name in labels_df['video'].unique():

    clean_name = video_name.replace(" ", "_") + "_cleaned.csv"
    csv_path   = os.path.join(csv_folder, clean_name)

    if not os.path.exists(csv_path):
        print(f"Skipping {video_name}, CSV not found")
        continue

    print(f"Processing: {video_name}")
    df = pd.read_csv(csv_path, index_col=0)

    # Get RF probabilities
    rf_out = get_rf_probabilities(video_name, df, pipeline, feature_cols)

    # Apply rule-based post-processing
    video_bouts = clean_rf_predictions(
        rf_out,
        fps= 25 ,
        min_bout_sec = 1,
        noise_gap_sec = 0.5,
        same_dir_gap_sec = 2,
        direction_window = 10
    )

    if len(bouts) > 0:
            video_bouts = finalise_bouts(
                video_bouts,
                fps = 25,
                merge_same_dir_gap_s = 5,
                end_isolation_gap_s = 30,
                min_end_bout_s = 3
            )
            video_bouts['video'] = video_name
            all_rf_bouts.append(video_bouts)

    rf_out['video'] = video_name
    all_rf_frames.append(rf_out)

    print(f"Final bouts: {len(video_bouts)}")
    if len(video_bouts) > 0:
        print(video_bouts[['start_frame', 'end_frame',
                            'duration_s', 'direction',
                            'direction_confidence']].to_string())

all_frames = pd.concat(all_rf_frames, ignore_index=True)
all_bouts = pd.concat(all_rf_bouts,  ignore_index=True) \
             if all_rf_bouts else pd.DataFrame()

In [ ]:
def extract_true_bouts_from_labels(labels_df, fps=25):
    """
    Extract bout-level records from frame-level labels_df - from manual annotations
    A bout = consecutive run of the same-direction grooming label
    """
    records = []

    for video_name, video_df in labels_df.groupby('video'):
        video_df = video_df.sort_values('frame').reset_index(drop=True)
        prev_label = None
        bout_start = None

        for _, row in video_df.iterrows():
            label = row['label']
            is_grooming = label not in ('no_grooming',) and not pd.isna(label)

            if is_grooming:
                if label != prev_label:
                    # Close previous bout if there was one
                    if prev_label not in ('no_grooming', None):
                        n_frames = row['frame'] - bout_start
                        records.append({
                            'video' : video_name,
                            'start_frame': bout_start,
                            'end_frame' : row['frame'] - 1,
                            'n_frames' : n_frames,
                            'duration_s' : n_frames / fps,
                            'direction' : prev_label,
                        })
                    bout_start = row['frame']
                prev_label = label

            else:
                # Close bout on no_grooming frame
                if prev_label not in ('no_grooming', None):
                    n_frames = row['frame'] - bout_start
                    records.append({
                        'video' : video_name,
                        'start_frame': bout_start,
                        'end_frame' : row['frame'] - 1,
                        'n_frames' : n_frames,
                        'duration_s': n_frames / fps,
                        'direction' : prev_label,
                    })
                prev_label = None
                bout_start = None

        # Close any open bout at end of video
        if prev_label not in ('no_grooming', None) and bout_start is not None:
            last_frame = video_df['frame'].iloc[-1]
            n_frames   = last_frame - bout_start + 1
            records.append({
                'video' : video_name,
                'start_frame': bout_start,
                'end_frame' : last_frame,
                'n_frames' : n_frames,
                'duration_s' : n_frames / fps,
                'direction' : prev_label,
            })

    return pd.DataFrame(records).reset_index(drop=True)


true_bouts = extract_true_bouts_from_labels(labels_df)
print(f"Total true bouts: {len(true_bouts)}")
print(f"\nDirection distribution:")
print(true_bouts['direction'].value_counts())
print(f"\nDuration summary (seconds):")
print(true_bouts['duration_s'].describe().round(1))
print(true_bouts.head(10).to_string())

In [ ]:
def evaluate_model_accuracy(true_bouts, pred_bouts, min_overlap=0.5):
    """
    Calculate the three key accuracy metrics:
    1. Bout count accuracy per video
    2. Bout duration accuracy (matched bouts)
    3. Direction accuracy (matched bouts)

    min_overlap: fraction of true bout that must be covered by a
                 predicted bout to count as detected
    """

    print(f"True bouts: {len(true_bouts)}")
    print(f"Pred bouts: {len(pred_bouts)}")

    # 1. Bout count accuracy
    videos = sorted(set(true_bouts['video'].unique()) | set(pred_bouts['video'].unique()))

    count_records = []
    for v in videos:
        t_n = len(true_bouts[true_bouts['video'] == v])
        p_n = len(pred_bouts[pred_bouts['video'] == v])
        count_records.append({
            'video' : v,
            'true_n_bouts': t_n,
            'pred_n_bouts': p_n,
            'count_diff' : p_n - t_n,
            'count_exact' : (t_n == p_n)
        })
    count_df = pd.DataFrame(count_records)

    print(f"\nBout count accuracy:")
    print(f"Exact match: {count_df['count_exact'].sum()}/{len(count_df)} "
          f"videos ({count_df['count_exact'].mean()*100:.1f}%)")
    print(f"Mean count diff: {count_df['count_diff'].mean():.2f} "
          f"(+ve = over-predicting)")

    # 2 & 3. Bout-level matching
    # For each true bout, find the predicted bout with most overlap
    match_records = []

    for _, t in true_bouts.iterrows():
        v_preds = pred_bouts[pred_bouts['video'] == t['video']]

        best_overlap = 0
        best_pred = None

        for _, p in v_preds.iterrows():
            overlap = (min(t['end_frame'], p['end_frame']) -
                       max(t['start_frame'], p['start_frame']))
            if overlap > best_overlap:
                best_overlap = overlap
                best_pred = p

        overlap_frac = best_overlap / t['n_frames'] \
                       if t['n_frames'] > 0 else 0

        detected = overlap_frac >= min_overlap

        if detected and best_pred is not None:
            dir_correct = (best_pred['direction'] == t['direction'])
            dur_error_s = abs(best_pred['duration_s'] - t['duration_s'])
            dur_error_pct = (dur_error_s / t['duration_s']) * 100
        else:
            dir_correct   = False
            dur_error_s   = np.nan
            dur_error_pct = np.nan

        match_records.append({
            'video' : t['video'],
            'true_start' : t['start_frame'],
            'true_end' : t['end_frame'],
            'true_duration_s' : t['duration_s'],
            'true_direction' : t['direction'],
            'pred_start' : best_pred['start_frame']
                                 if best_pred is not None else np.nan,
            'pred_end' : best_pred['end_frame']
                                 if best_pred is not None else np.nan,
            'pred_duration_s' : best_pred['duration_s']
                                 if best_pred is not None else np.nan,
            'pred_direction' : best_pred['direction']
                                 if best_pred is not None else None,
            'overlap_frac' : overlap_frac,
            'detected' : detected,
            'dir_correct' : dir_correct,
            'dur_error_s' : dur_error_s,
            'dur_error_pct' : dur_error_pct,
        })

    match_df = pd.DataFrame(match_records)

    detected = match_df[match_df['detected']]
    n_true = len(match_df)
    n_detected = len(detected)

    print(f"\nBout detection accuracy: ")
    print(f"Bouts detected (>={min_overlap*100:.0f}% overlap): "
          f"{n_detected}/{n_true} ({n_detected/n_true*100:.1f}%)")
    print(f"Bouts missed: {n_true - n_detected}")

    print(f"\nDuration accuracy of detection bouts:")
    print(f"Mean duration error:   {detected['dur_error_s'].mean():.1f}s")
    print(f"Median duration error: {detected['dur_error_s'].median():.1f}s")
    print(f"Mean % error:          {detected['dur_error_pct'].mean():.1f}%")
    print(f"Median % error:        {detected['dur_error_pct'].median():.1f}%")

    print(f"\nDirection accuracy of detection bouts:")
    n_dir_correct = detected['dir_correct'].sum()
    print(f"Correct: {n_dir_correct}/{n_detected} "
          f"({n_dir_correct/n_detected*100:.1f}%)")
    print(f"Wrong: {n_detected - n_dir_correct}/{n_detected} "
          f"({(n_detected-n_dir_correct)/n_detected*100:.1f}%)")

    # Direction accuracy by video
    dir_by_video = detected.groupby('video')['dir_correct'].mean()
    print(f"\nDirection accuracy per video:")
    print(f"Mean: {dir_by_video.mean()*100:.1f}%")
    print(f"Median: {dir_by_video.median()*100:.1f}%")
    print(f"Videos with 100% dir accuracy: "
          f"{(dir_by_video == 1.0).sum()}/{len(dir_by_video)}")
    print(f"Videos with 0% dir accuracy: "
          f"{(dir_by_video == 0.0).sum()}/{len(dir_by_video)}")
    
    return count_df, match_df

In [ ]:
# Get metrics
count_df, match_df = evaluate_model_accuracy(
    true_bouts = true_bouts,
    pred_bouts = all_bouts,
    min_overlap   = 0.5
)

In [ ]:
import matplotlib.patches as mpatches

def plot_video_predictions(video_name, all_frames, all_bouts, true_bouts):

    direction_colours = {'A_gr_B': 'steelblue', 'B_gr_A': 'coral'}

    frames = all_frames[all_frames['video'] == video_name].sort_values('frame')
    pred = all_bouts[all_bouts['video'] == video_name]
    true = true_bouts[true_bouts['video'] == video_name]

    fig, axes = plt.subplots(3, 1, figsize=(12, 4), sharex=True)

    # Panel 1: RF probability for both grooming directions
    axes[0].plot(frames['frame'], frames['p_A_gr_B'], color='steelblue', linewidth=0.8, alpha=0.8, label='P(A grooms B)')
    axes[0].plot(frames['frame'], frames['p_B_gr_A'], color='coral', linewidth=0.8, alpha=0.8, label='P(B grooms A)')
    axes[0].axhline(0.6, color='red', linestyle='--', linewidth=0.8)
    axes[0].set_ylabel('P(grooming)')
    axes[0].set_ylim(0, 1)
    axes[0].set_xlim(0, )
    axes[0].set_title(f'{video_name}')
    axes[0].legend(loc='upper right', fontsize=8)

    # Panel 2: Post-processed predicted bouts
    for _, bout in pred.iterrows():
        color = direction_colours.get(bout['direction'], 'grey')
        axes[1].axvspan(bout['start_frame'], bout['end_frame'], alpha=0.6, color=color)
    axes[1].set_ylabel('Predicted')
    axes[1].set_yticks([])

    # Panel 3: True bouts
    for _, bout in true.iterrows():
        color = direction_colours.get(bout['direction'], 'grey')
        axes[2].axvspan(bout['start_frame'], bout['end_frame'], alpha=0.6, color=color)
    axes[2].set_ylabel('True')
    axes[2].set_yticks([])
    axes[2].set_xlabel('Frame')

    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='steelblue', label='A grooms B'),
        mpatches.Patch(color='coral', label='B grooms A'),
    ]
    axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

for video_name in sorted(labels_df['video'].unique()):
    plot_video_predictions(
        video_name = video_name,
        all_frames = all_frames,
        all_bouts  = all_bouts,
        true_bouts = true_bouts
    )

# Complete run through of the full video dataset

# 8. Run videos through YOLO & Bytetrack

In [ ]:
# Settings
yolo_path = r"DATA_DIR\YOLO_training\yolo_monkey\weights\best.pt"
video_folder = r"DATA_DIR\complete_videos"
bbox_output = r"DATA_DIR\complete_box_tracks"
os.makedirs(bbox_output, exist_ok=True)

In [ ]:
# Run full video dataset through step 2

yolo_model = YOLO(yolo_path)
videos = glob.glob(os.path.join(video_folder, "*.mp4"))
print(f"{len(videos)} videos\n")

for i, video_path in enumerate(videos, start=1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    out_path = os.path.join(bbox_output,video_name.replace(" ", "_") + "_boxes.csv")

    if os.path.exists(out_path):
        continue

    print(f"Processing: {video_name} ({i}/{len(videos)})")

    try:
        frame_boxes, track_counts, track_centroids = run_yolo_bytetrack(video_path, yolo_model)
        n_frames = len(frame_boxes)
        print(f"Frames: {n_frames}, Tracks: {len(track_counts)}")

        if len(track_counts) < 2:
            print("Fewer than 2 tracks detected, skipping")
            continue

        chain_stats = stitch_tracks(track_centroids, track_counts, frame_boxes)
        frame_A, frame_B = build_AB_timelines(chain_stats, frame_boxes, n_frames)
        frame_A = interpolate_boxes(frame_A, n_frames)
        frame_B = interpolate_boxes(frame_B, n_frames)

        a_final = sum(1 for b in frame_A if b is not None)
        b_final = sum(1 for b in frame_B if b is not None)
        print(f"A coverage (final): {a_final/n_frames*100:.1f}%")
        print(f"B coverage (final): {b_final/n_frames*100:.1f}%")

        save_box_csv(frame_A, frame_B, n_frames, out_path)
        print(f"Saved: {os.path.basename(out_path)}\n")

    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        traceback.print_exc()
        continue

# 9. Preprocess DLC data and assign A and B (same as step 3)

In [ ]:
# Setting
h5_folder = r"DATA_DIR\complete_h5"
h5_files = glob.glob(os.path.join(h5_folder, "*.h5"))
box_folder = r"DATA_DIR\complete_box_tracks"
csv_folder = r"DATA_DIR\complete_csv"
os.makedirs(csv_folder, exist_ok=True)
print(f"{len(h5_files)} h5 files")

In [ ]:
# Run all videos through step 3
for f in h5_files:
    filename = os.path.basename(f)
    video_name = filename.split("_superanimal")[0]
    box_path = os.path.join(box_folder, video_name.replace(" ", "_") + "_boxes.csv")
    out_path = os.path.join(csv_folder, video_name.replace(" ", "_") + "_cleaned.csv")

    if not os.path.exists(box_path):
        print(f"No box CSV for {video_name}")
        continue

    print(f"Processing: {video_name}")

    try:
        df_raw  = pd.read_hdf(f)
        scorer  = df_raw.columns.get_level_values(0).unique()[0]
        animals = df_raw.columns.get_level_values(1).unique().tolist()
        df_raw  = df_raw.replace(-1.0, np.nan)

        box_df  = pd.read_csv(box_path, index_col=0)
        n_frames = len(df_raw)

        kp_data = assign_keypoints_boxes(df_raw, scorer, animals, box_df)
        out_df = pd.DataFrame(kp_data)
        out_df.index.name = 'frame'

        out_df = interpolate_keypoints(out_df)
        out_df = apply_kalman(out_df, 'A')
        out_df = apply_kalman(out_df, 'B')

        # Coverage summary
        for identity in ['A', 'B']:
                col = f'{identity}_back_middle_x'
                if col in out_df.columns:
                    cov = out_df[col].notna().mean() * 100
                    print(f"{identity} coverage: {cov:.1f}")

        out_df.to_csv(out_path)
        print(f"Saved: {os.path.basename(out_path)}\n")
    
    except Exception as e:
         import traceback
         print(f"Error: {e}")
         traceback.print_exc()

# 10. Quality Control

1. Compute per-video quality scores from:
    Coverage of A and B identity assignment across video
    Stability of centroids across frames
    DLC likelihood scores
    Coverage of confident keypoints across videos
2. Assess quality distribution
3. Filter out low quality videos based on quality distribution

In [ ]:
def compute_video_quality(box_path, kp_path):
    """
    Compute per-video quality scores from box tracks and keypoints
    
    Returns dict of quality metrics:
    
    Box-level (from YOLO/ByteTrack):
        both_coverage : fraction of frames where both A and B detected
        A_coverage : fraction of frames where A detected
        B_coverage : fraction of frames where B detected
        A_stability : median frame-to-frame centroid jump for A (pixels)
        B_stability : median frame-to-frame centroid jump for B (pixels)
        
    Keypoint-level (from DLC):
        A_mean_likelihood : mean DLC likelihood for A on detected frames
        B_mean_likelihood : mean DLC likelihood for B on detected frames
        A_kp_coverage : fraction of frames with confident A keypoints
        B_kp_coverage : fraction of frames with confident B keypoints
        both_kp_coverage : fraction of frames with both identities detected
        
    Combined:
        quality_score : overall 0-1 score combining all metrics
    """
    box_df = pd.read_csv(box_path, index_col=0)
    kp_df = pd.read_csv(kp_path,  index_col=0)
    n = len(box_df)

    # Box metrics
    a_present  = box_df['A_x1'].notna()
    b_present  = box_df['B_x1'].notna()
    both_boxes = (a_present & b_present)

    # Stability: median centroid jump frame to frame
    # Large jumps = identity swaps or noisy tracking
    a_jump = np.sqrt(box_df['A_cx'].diff()**2 + box_df['A_cy'].diff()**2).median()
    b_jump = np.sqrt(box_df['B_cx'].diff()**2 + box_df['B_cy'].diff()**2).median()

    # Keypoint metrics
    # Coverage: frames where identity has at least one confident keypoint
    a_kp_present = pd.Series(False, index=kp_df.index)
    b_kp_present = pd.Series(False, index=kp_df.index)
    a_liks, b_liks = [], []

    for bp in all_parts:
        a_l_col = f'A_{bp}_l'
        b_l_col = f'B_{bp}_l'
        if a_l_col in kp_df.columns:
            a_conf = kp_df[a_l_col] > 0
            a_kp_present |= a_conf
            a_liks.extend(kp_df.loc[a_conf, a_l_col].tolist())
        if b_l_col in kp_df.columns:
            b_conf = kp_df[b_l_col] > 0
            b_kp_present |= b_conf
            b_liks.extend(kp_df.loc[b_conf, b_l_col].tolist())

    both_kp = (a_kp_present & b_kp_present)

    a_mean_lik = np.mean(a_liks) if a_liks else 0.0
    b_mean_lik = np.mean(b_liks) if b_liks else 0.0

    # Combined quality score
    # Weighted combination of key metrics
    # Both coverage is most important - need both animals tracked for direction classifcation
    # Jumpy tracks = identity swaps
    # Low likelihood = noisy features

    # Normalise stability: 0 = very jumpy, 1 = very stable
    # good tracking: <50px jumps, bad: >200px
    a_stab = np.clip(1 - (a_jump / 200), 0, 1)
    b_stab = np.clip(1 - (b_jump / 200), 0, 1)

    quality_score = (
        0.35 * both_kp.mean() + # most important: both tracked
        0.20 * a_kp_present.mean() +  # A individually tracked
        0.20 * b_kp_present.mean() +  # B individually tracked
        0.10 * a_mean_lik + # DLC confidence A
        0.10 * b_mean_lik + # DLC confidence B
        0.025 * a_stab + # stability A
        0.025 * b_stab # stability B
    )

    return {
        'n_frames' : n,
        'both_box_coverage': both_boxes.mean(),
        'A_box_coverage' : a_present.mean(),
        'B_box_coverage' : b_present.mean(),
        'A_stability_px' : a_jump,
        'B_stability_px' : b_jump,
        'A_kp_coverage' : a_kp_present.mean(),
        'B_kp_coverage' : b_kp_present.mean(),
        'both_kp_coverage': both_kp.mean(),
        'A_mean_likelihood': a_mean_lik,
        'B_mean_likelihood': b_mean_lik,
        'quality_score' : quality_score,
    }

In [ ]:
# Run on all videos
box_files = glob.glob(os.path.join(box_folder, "*_boxes.csv"))
print(f"Found {len(box_files)} box CSVs")

quality_records = []

for box_path in sorted(box_files):
    box_name = os.path.basename(box_path).replace("_boxes.csv", "")
    csv_path = os.path.join(csv_folder, box_name + "_cleaned.csv")
    video_name = box_name.replace("_", " ")

    if not os.path.exists(kp_path):
        print(f"No keypoint CSV for {box_name}")
        continue

    try:
        metrics = compute_video_quality(box_path, kp_path)
        metrics['video'] = video_name
        quality_records.append(metrics)
    except Exception as e:
        print(f"Error {box_name}: {e}")
        continue

quality_df = pd.DataFrame(quality_records).set_index('video')
quality_df = quality_df.sort_values('quality_score')

print(f"\nQuality score distribution:")
print(quality_df['quality_score'].describe().round(3))

In [ ]:
# Plot histogram of wuality distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

metrics_to_plot = [
    ('quality_score', 'Overall quality score'),
    ('both_kp_coverage', 'Both identities detected (fraction)'),
    ('A_kp_coverage', 'A keypoint coverage'),
    ('B_kp_coverage', 'B keypoint coverage'),
    ('A_mean_likelihood','A mean DLC likelihood'),
    ('B_mean_likelihood','B mean DLC likelihood'),
]

for ax, (col, title) in zip(axes, metrics_to_plot):
    ax.hist(quality_df[col], bins=30, color='steelblue', alpha=0.7,
            edgecolor='white')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Videos')
    ax.grid(alpha=0.3)

plt.suptitle('Video Quality Distribution', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Apply threshold and filter

# Filter by quality threshold
fixed_threshold = 0.3 # quality score threshold
high_quality = quality_df[quality_df['quality_score'] >= fixed_threshold]

# Filter by keypoint coverage threshold
kp_threshold = 0.1 # quality score threshold
high_quality = quality_df[(quality_df['both_kp_coverage'] >= kp_threshold)]
low_quality = quality_df[(quality_df['quality_score'] < fixed_threshold) & (quality_df['both_kp_coverage'] < kp_threshold)]

print(f"\nQuality threshold: {fixed_threshold}, Both keypoint coverage threshold: {kp_threshold}")
print(f"High quality: {len(high_quality)} videos")
print(f"Low quality: {len(low_quality)} videos")

print(f"\nLow quality videos (excluded):")
print(low_quality[['quality_score', 'both_kp_coverage',
                    'A_mean_likelihood', 'B_mean_likelihood',
                    'n_frames']].round(3).to_string())

# List of videos to use
high_quality_videos = high_quality.index.tolist()

# 11. Feature extraction (same as step 5)

In [ ]:
# Run step 5 on all videos

csv_files = glob.glob(os.path.join(csv_folder, "*_cleaned.csv"))

all_features = []

for csv_path in sorted(csv_files):

    video_name = os.path.basename(csv_path).replace("_cleaned.csv", "").replace("_", " ")
    print(f"Processing: {video_name}")

    try:
        df = pd.read_csv(csv_path, index_col=0)
        features = feature_extraction(df).reset_index(drop=True)
        features['video'] = video_name
        features['frame'] = features.index
        all_features.append(features)

    except Exception as e:
        print(f"Error: {e}")
        continue

features_df = pd.concat(all_features, ignore_index=True)
print(f"\nFeatures shape: {features_df.shape}")
print(f"Videos processed: {features_df['video'].nunique()}")

# 12. Classification & Postprocessing

1. Run classification pipeline on complete video dataset
2. Apply postprocessing steps
3. Save bout predictions as csv

In [ ]:
all_predictions = []
all_bouts = []

for video_name, video_features in features_df.groupby('video'):
    print(f"Processing: {video_name}")

    try:
        # Align features to training columns
        X = video_features.reindex(columns=feature_cols).copy()

        # Predict
        proba = pipeline.predict_proba(X)

        # Build per-frame output
        proba_df = pd.DataFrame(proba, columns=pipeline.classes_)
        proba_df['video'] = video_name
        proba_df['frame'] = video_features['frame'].values
        proba_df['p_grooming'] = proba_df[[c for c in pipeline.classes_ if c != 'no_grooming']].sum(axis=1)
        proba_df['p_A_gr_B'] = proba_df.get('A_gr_B', 0)
        proba_df['p_B_gr_A'] = proba_df.get('B_gr_A', 0)

        all_predictions.append(proba_df)

        # Post-process into bouts
        bouts = clean_rf_predictions(
            proba_df,
            fps = 25,
            min_bout_sec = 1,
            noise_gap_sec = 0.5,
            same_dir_gap_sec = 2,
            direction_window = 10
        )

        if len(bouts) > 0:
            bouts = finalise_bouts(
                bouts,
                fps = 25,
                merge_same_dir_gap_s = 5.0,
                end_isolation_gap_s = 30.0,
                min_end_bout_s= 3.0
            )
            bouts['video'] = video_name
            all_bouts.append(bouts)

        print(f"Frames: {len(X)}, Bouts: {len(bouts)}")

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()
        continue

In [ ]:
# Combine
all_frames_new = pd.DataFrame(pd.concat(all_predictions, ignore_index=True))
all_bouts_new  = pd.DataFrame(pd.concat(all_bouts, ignore_index=True))

# Save
output_folder = r"DATA_DIR\complete_predictions"
os.makedirs(output_folder, exist_ok=True)

all_frames_new.to_csv(os.path.join(output_folder, 'all_frame_predictions.csv'), index=False)
all_bouts_new.to_csv(os.path.join(output_folder, 'all_bout_predictions.csv'), index=False)
print(f"Saved to: {output_folder}")